In [7]:
# -----------------------------------------------------------------------------
# 1. 导入所需库与全局配置
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # 非交互后端：plt.show() 不阻塞，图仅存盘
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体，防止图表中文乱码
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

# 设置 Pandas 显示选项，便于查看宽表
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# 固定随机种子（全项目统一）
SEED = 42
np.random.seed(SEED)
TARGET = 'y'  # 目标变量名称
# =============================================================================
# 步骤 ①：业务理解与数据探索（EDA）
# 目的：全面了解数据分布、缺失情况、异常值及关键变量与违约率的关系
# =============================================================================
print('=' * 70)
print('① 业务理解与数据探索（EDA）')
print('=' * 70)

# ---------- 1.1 加载数据与目标变量概览 ----------
df_raw = pd.read_csv('lc_loan_data.csv')
print('数据形状:', df_raw.shape)
print('目标变量 y：1=违约(坏) 0=履约(好)')
print(df_raw[TARGET].value_counts().rename({0: '好客户', 1: '坏客户'}).to_string())
print('坏账率: %.2f%%' % (df_raw[TARGET].mean() * 100))
print()

# ---------- 1.2 全量缺失值诊断（含缺失为0的列） ----------
# 目的：明确每列缺失率，为后续缺失处理提供依据
miss = df_raw.isna().sum()
miss_table = (miss.rename('缺失数').to_frame()
              .assign(**{'缺失率%': lambda x: (x['缺失数'] / len(df_raw) * 100).round(2)}))
print('缺失值一览（全部 %d 列的缺失率，含缺失为 0 的列）：' % len(miss_table))
print(miss_table.to_string())
print()

# ---------- 1.3 数值型变量描述统计 ----------
# 为什么补全量：单变量分布不能只看少数几个变量，应对所有候选数值变量给描述统计，
# 便于全面发现极端值/偏态/量纲，为后续清洗与分箱提供依据。
EDA_NUMERIC = ['loan_amnt', 'funded_amnt', 'int_rate', 'installment', 'emp_length',
               'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq',
               'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
               'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal',
               'total_rev_hi_lim', 'mths_since_earliest_cr_line']
print('单变量分布（全量数值变量 describe）：')
print(df_raw[EDA_NUMERIC].describe().round(2).to_string())
print()

# ---------- 1.4 类别型变量频数统计 ----------
EDA_CAT = ['grade', 'term', 'home_ownership', 'verification_status',
           'purpose', 'initial_list_status', 'addr_state', 'sub_grade']
print('单变量分布（类别变量频数统计，含各档样本数）：')
for c in EDA_CAT:
    vc = df_raw[c].astype(str).value_counts()
    top = vc.head(12)
    line = ', '.join(f'{k}={v}' for k, v in top.items())
    if len(vc) > 12:
        line += f', ...(共{len(vc)}类)'
    print(f'  {c}: {line}')
print()

# ---------- 1.5 类别变量频数条形图（用于PPT展示） ----------
# 只画核心入模类别变量，直观看出各档分布与稀有类
cat_plot = ['grade', 'term', 'home_ownership', 'verification_status', 'purpose', 'initial_list_status']
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), cat_plot):
    vc = df_raw[c].astype(str).value_counts()
    if len(vc) > 10:            # 高基数变量（如 purpose 14 类）只画前 10 类，避免柱子过密
        vc = vc.head(10)
    sns.barplot(x=vc.index, y=vc.values, ax=ax, color='steelblue')
    ax.set_title(c, fontsize=12)
    ax.set(ylabel='样本数', xlabel=None)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
plt.suptitle('类别变量频数分布', fontsize=14)
plt.tight_layout()
plt.savefig('fig_categorical.png', dpi=120)

# ---------- 1.6 重点异常值识别（基于数据字典和业务常识） ----------
print('异常值识别：')
print('  revol_util > 100% 的样本数:', (df_raw['revol_util'] > 100).sum())
print('  annual_inc 极端大值（99% 分位以上）:', (df_raw['annual_inc'] > df_raw['annual_inc'].quantile(0.99)).sum(),
      '，最大:', df_raw['annual_inc'].max())
print('  tot_coll_amt / tot_cur_bal / total_rev_hi_lim 三者同时缺失:',
      df_raw[['tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim']].isna().all(axis=1).sum())
print()

# ---------- 1.7 关键变量与违约率的初步关系（验证业务直觉） ----------
print('关键变量 × 违约率（初步关系）：')
print('  term 36个月 坏账率: %.2f%% | term 60个月 坏账率: %.2f%%'
      % (df_raw.loc[df_raw['term'] == 36, TARGET].mean() * 100,
         df_raw.loc[df_raw['term'] == 60, TARGET].mean() * 100))
grade_br = df_raw.groupby('grade', observed=True)[TARGET].agg(['mean', 'count'])
grade_br['mean'] = (grade_br['mean'] * 100).round(2)
print('  grade × 坏账率：')
print(grade_br.rename(columns={'mean': '坏账率%', 'count': '样本数'}).to_string())
print('  int_rate 与 grade 相关系数（有序映射后）:',
      round(df_raw['int_rate'].corr(df_raw['grade'].map({g: i for i, g in enumerate(sorted(df_raw['grade'].unique()))})), 4))
print()

# ---------- 1.8 清洗前箱线图（可视化极端值分布） ----------
_num_cols = ['loan_amnt', 'installment', 'annual_inc', 'dti', 'revol_bal',
             'revol_util', 'total_acc', 'mths_since_earliest_cr_line']
fig, axes = plt.subplots(2, 4, figsize=(18, 6))
for ax, c in zip(axes.ravel(), _num_cols):
    sns.boxplot(y=df_raw[c], ax=ax)
    ax.set_title(c, fontsize=10)
    ax.set(ylabel=None)
plt.suptitle('清洗前箱线图（可见 annual_inc / revol_util / revol_bal 存在极端值）', fontsize=13)
plt.tight_layout()
plt.savefig('fig1_eda_boxplot_before.png', dpi=120)
# ============================================================
# ② 数据清洗（fit/apply 分离：中位数与截尾阈值只在训练集 fit）
# ============================================================
print('=' * 70)
print('② 数据清洗')
print('=' * 70)


def clean_fit(df_train):
    """在训练集上学习填充中位数与截尾阈值，返回参数。

    为什么只在训练集 fit：中位数/分位点是用数据「学」出来的量，只能在训练集计算，
    测试集只套用。
    """
    return {
        'revol_util_med': df_train['revol_util'].median(),
        'tot_coll_amt_med': df_train['tot_coll_amt'].median(),
        'tot_cur_bal_med': df_train['tot_cur_bal'].median(),
        'total_rev_hi_lim_med': df_train['total_rev_hi_lim'].median(),
        'inc_cap': df_train['annual_inc'].quantile(0.99),
    }


def clean_apply(df_in, p):
    """套用训练集学到的参数做填充与截尾。"""
    df = df_in.copy()

    # ── 缺失值处理 ──
    # 为什么不用 dropna 扔样本：缺失本身有信息（如「从未逾期」的客户才缺 mths_since_last_delinq），
    # 直接丢弃会损失大量样本并引入偏差，所以改用「有业务含义的填充」保留信息。
    # 为什么金额类用中位数：金额变量右偏严重、存在极端值，中位数对极端值稳健、
    # 更能代表「典型客户」水平，均值会被少数巨款客户拉高。
    df['mths_since_last_delinq'] = df['mths_since_last_delinq'].fillna(-1)  # 56%缺失=从未逾期，用 -1 单独标记，分箱时自成「从未逾期」档
    df['revol_util'] = df['revol_util'].fillna(p['revol_util_med'])          # 32个缺失(0.06%)，金额类中位数填充
    df['collections_12_mths_ex_med'] = df['collections_12_mths_ex_med'].fillna(0)  # 99.5%为0，缺失大概率也是无催收，填 0
    df['tot_coll_amt'] = df['tot_coll_amt'].fillna(p['tot_coll_amt_med'])    # 金额类，中位数填充
    df['tot_cur_bal'] = df['tot_cur_bal'].fillna(p['tot_cur_bal_med'])        # 金额类，中位数填充
    df['total_rev_hi_lim'] = df['total_rev_hi_lim'].fillna(p['total_rev_hi_lim_med'])  # 金额类，中位数填充（三者同批缺失，同一批无余额信息客户）

    # ── 异常值处理 ──
    # 为什么 revol_util 截尾到 100：循环额度使用率按定义上限就是 100%，>100% 属于口径/录入异常，
    # 截到 100 更符合业务含义，且比删除更保守（只改值、不丢样本）。
    df['revol_util'] = df['revol_util'].clip(upper=100)
    # 为什么极端收入用 winsorize(99%分位截尾) 而非删除：收入存在数百万美元极端值，
    # 删除会损失样本且带主观性；winsorize 到 99% 分位既保留样本、又抑制极端值对分箱/WOE 的扰动。
    df['annual_inc'] = df['annual_inc'].clip(upper=p['inc_cap'])
    return df


# ============================================================
# ③ 特征工程（衍生特征 ≥3；同样 fit/apply 分离）
# ============================================================
print('=' * 70)
print('③ 特征工程（衍生特征）')
print('=' * 70)

# 为什么构造比值特征：绝对金额（贷款额/循环欠款）在不同收入人群间不可比，
# 除以年收入后变成「相对负担/杠杆」，既消除量纲差异、业务含义也更清晰（还贷压力、杠杆水平）。
DERIVED = ['repay_burden', 'loan_to_income', 'revol_bal_to_income',
           'tot_bal_to_income', 'credit_line_util']


def _derive(df):
    inc = df['annual_inc'].replace(0, np.nan)
    df['repay_burden'] = df['installment'] * 12 / inc         # 年还款负担率
    df['loan_to_income'] = df['loan_amnt'] / inc               # 贷款额收入比
    df['revol_bal_to_income'] = df['revol_bal'] / inc          # 循环负债收入比
    df['tot_bal_to_income'] = df['tot_cur_bal'] / inc          # 总余额收入比
    df['credit_line_util'] = df['revol_bal'] / df['total_rev_hi_lim'].replace(0, np.nan)  # 循环额度使用率
    return df


def feat_fit(df_train):
    """在训练集上计算衍生特征的截尾/填充参数。"""
    tmp = _derive(df_train.copy())
    p = {}
    for c in DERIVED:
        s = tmp[c].replace([np.inf, -np.inf], np.nan)
        p[f'{c}_cap'] = s.quantile(0.99)
        p[f'{c}_med'] = s.median()
    return p


def feat_apply(df_in, p):
    """构造衍生特征，并用训练集参数做极端值处理。"""
    df = _derive(df_in.copy())
    for c in DERIVED:
        df[c] = df[c].replace([np.inf, -np.inf], np.nan)
        df[c] = df[c].clip(upper=p[f'{c}_cap'])
        df[c] = df[c].fillna(p[f'{c}_med'])
    return df


print('衍生特征（5 个，≥3）及业务含义（前 3 个为题目建议方向，后 2 个为自行探索）：')
print('  repay_burden       年还款负担率 = 月供×12/年收入，越高还贷压力越大、越易违约')
print('  loan_to_income     贷款额收入比 = 申请金额/年收入，衡量贷款杠杆，越高越激进')
print('  revol_bal_to_income 循环负债收入比 = 循环欠款/年收入，存量循环负债的相对负担')
print('  tot_bal_to_income  总余额收入比 = 账户总余额/年收入，总负债规模相对收入')
print('  credit_line_util   循环额度使用率 = 循环欠款/授信总额度（交叉特征：用余额与额度两列重构使用率，与平台 revol_util 互为印证）')
print()

# ============================================================
# ④ 数据集划分（70/30，固定 seed）—— 先划分，再拟合清洗/特征参数
# ============================================================
print('=' * 70)
print('④ 数据集划分（70/30，固定 seed=42）')
print('=' * 70)

# 为什么 70/30 + stratify + 固定 seed：70% 保证模型学得充分、30% 保证评估可靠；
# stratify 让训练/测试坏账率一致（本数据 19.25%），避免抽样偏斜；固定 seed 保证可复现。
# 为什么先划分再清洗：填充中位数/截尾阈值/分箱边界等「用数据学出来的量」都只能在训练集计算，
# 测试集只套用，否则等于把测试集信息泄漏进模型（防泄漏是评分卡第一纪律）。
train_raw, test_raw = train_test_split(df_raw, test_size=0.3, stratify=df_raw[TARGET], random_state=SEED)

# 清洗：只在训练集 fit，训练/测试统一 apply
clean_p = clean_fit(train_raw)
train_c = clean_apply(train_raw, clean_p)
test_c = clean_apply(test_raw, clean_p)

# 特征工程：只在训练集 fit，训练/测试统一 apply
feat_p = feat_fit(train_c)
train = feat_apply(train_c, feat_p)
test = feat_apply(test_c, feat_p)

print(f'训练集 {len(train)} 行 | 测试集 {len(test)} 行')
print('训练集坏账率 %.2f%% | 测试集坏账率 %.2f%%' % (train[TARGET].mean() * 100, test[TARGET].mean() * 100))

# ── 缺失值处理审计──
# 原则：每个缺失字段都必须明确归属三类之一——①单独缺失/特殊箱 ②业务含义箱 ③训练集统计量填充；
#       训练/测试统一用训练集学到的规则，保证一致（防泄漏）。
print('缺失值处理策略（每列明确归属）：')
for col, strat in [
    ('mths_since_last_delinq', '填 -1 → 分箱时 -1 单独成「从未逾期」档（单独缺失/特殊箱）'),
    ('collections_12_mths_ex_med', '填 0 → 0 单独成箱（业务含义：缺失视为无催收）'),
    ('revol_util', '训练集中位数填充（训练集统计量）'),
    ('tot_coll_amt / tot_cur_bal / total_rev_hi_lim', '训练集中位数填充（训练集统计量，三者同批缺失）'),
]:
    print(f'  {col}: {strat}')
# 断言闭环：清洗+特征工程后训练/测试不得再有任何 NaN（否则 apply_bins 会把 NaN 误当字符串 "nan"）
assert train.isna().sum().sum() == 0 and test.isna().sum().sum() == 0, '清洗后仍存在缺失值！'
print('审计通过：清洗+特征工程后，训练集与测试集均无缺失值（NaN=0），缺失已按上述策略显式处理、训练/测试一致。')
print('  annual_inc 截尾阈值(训练集): %.0f' % clean_p['inc_cap'])
print()

fig, axes = plt.subplots(2, 4, figsize=(18, 6))
for ax, c in zip(axes.ravel(), _num_cols):
    sns.boxplot(y=train[c], ax=ax)
    ax.set_title(c, fontsize=10)
    ax.set(ylabel=None)
plt.suptitle('清洗后箱线图（训练集，极端值已处理）', fontsize=13)
plt.tight_layout()
plt.savefig('fig2_eda_boxplot_after.png', dpi=120)

# ============================================================
# ⑤ 分箱（训练集定边界，测试集套用）
# ============================================================
print('=' * 70)
print('⑤ 分箱')
print('=' * 70)

# 特征池：剔除 funded_amnt(与loan_amnt相关0.997)、sub_grade(与grade冗余)、issue_month(时间不入模)；
#         int_rate 与 grade 高度相关(0.95)，二者保留其一供评分卡
cat_cols = ['term', 'grade', 'home_ownership', 'verification_status',
            'purpose', 'addr_state', 'initial_list_status']
cont_cols = ['loan_amnt', 'installment', 'emp_length', 'annual_inc', 'dti',
             'open_acc', 'revol_bal', 'revol_util', 'total_acc',
             'mths_since_earliest_cr_line', 'int_rate'] + DERIVED
zero_cols = ['delinq_2yrs', 'inq_last_6mths', 'pub_rec', 'acc_now_delinq',
             'collections_12_mths_ex_med', 'tot_coll_amt']
special_cols = ['mths_since_last_delinq']  # -1 = 从未逾期

feature_cols = cat_cols + cont_cols + zero_cols + special_cols
print('候选特征数:', len(feature_cols))


# 为什么分箱：把连续变量切成有序区间，捕捉「不同区间对违约率的不同影响」（非线性），且对异常值稳健；
# WOE 和评分卡必须建立在离散档位上。
# 为什么等频分箱（而非等宽）：保证每箱样本数大致相等，避免某箱样本过少导致 WOE 不稳定。
def _q_edges(s, n_bins):
    qs = s.quantile([i / n_bins for i in range(1, n_bins)])
    return sorted(set(np.round(qs.values, 6)))


def build_bins(train_s, kind, n_bins=5, min_freq=0.02):
    """返回分箱规则。kind ∈ {'cont','zero','special','cat'}。"""
    if kind == 'cont':
        return {'edges': sorted(set([-np.inf] + _q_edges(train_s, n_bins) + [np.inf])), 'cat_map': None}
    if kind == 'zero':  # 为什么 0 单独成箱：这些变量绝大多数为 0，「0」是有业务含义的状态（无逾期/无查询），须与正数分开
        pos = train_s[train_s > 0]
        return {'edges': sorted(set([-np.inf, 0.0] + _q_edges(pos, max(n_bins - 1, 2)) + [np.inf])), 'cat_map': None}
    if kind == 'special':  # 特殊值(-1=从未逾期)单独成箱
        rest = train_s[train_s >= 0]
        return {'edges': sorted(set([-np.inf, -0.5] + _q_edges(rest, n_bins) + [np.inf])), 'cat_map': None}
    if kind == 'cat':  # 为什么稀有类合并：样本太少的类别 WOE 估计不可靠（方差大），合并可避免过拟合
        s = train_s.astype(str)
        vc = s.value_counts()
        keep = set(vc[vc / len(s) >= min_freq].index)
        return {'edges': None, 'cat_map': {v: (v if v in keep else '其他') for v in vc.index}}


def apply_bins(s, rule):
    """把分箱规则套到 Series 上，返回类别标签（字符串）。

    缺失值约定：缺失已在清洗阶段全部显式处理（见「缺失值处理审计」的 assert），
    故这里假定输入无 NaN；若真出现 NaN，说明上游清洗有遗漏，应回查而非静默吞掉。
    """
    if rule['cat_map'] is not None:
        return s.astype(str).map(rule['cat_map']).fillna('其他')
    return pd.cut(s, rule['edges'], include_lowest=True).astype(str)


kind_map = {}
for c in cat_cols:
    kind_map[c] = 'cat'
for c in cont_cols:
    kind_map[c] = 'cont'
for c in zero_cols:
    kind_map[c] = 'zero'
for c in special_cols:
    kind_map[c] = 'special'

NO_MERGE = {'grade'}  # 有序且有业务含义的小基数变量，不合并稀有类
bins_map = {c: build_bins(train[c], kind_map[c], min_freq=(0.0 if c in NO_MERGE else 0.02))
            for c in feature_cols}
print('分箱规则已在训练集上建立。示例：')
print('  term:', bins_map['term']['cat_map'])
print('  delinq_2yrs(0单独成箱) 边界:', bins_map['delinq_2yrs']['edges'])
print('  mths_since_last_delinq(-1单独成箱) 边界:', bins_map['mths_since_last_delinq']['edges'])
print()

# =============================================================================
# 步骤 ⑥：WOE 与 IV（证据权重与信息价值）
# 目的：将原始特征转换为 WOE 编码，通过 IV 筛选有效特征，并进行共线性诊断
# 关键原则：所有 WOE/IV 计算仅在训练集上进行，测试集仅使用 transform
# =============================================================================
print('=' * 70)
print('⑥ WOE 与 IV')
print('=' * 70)


# ---------- 6.1 WOE/IV 计算函数定义 ----------
# 为什么用 WOE：把原始取值换成「该档位的风险强度」，天然处理非线性/单调性，逻辑回归系数也有业务含义。
# 为什么 +0.5 平滑：某档若坏样本数为 0，取对数会除零/无穷，加平滑项保证数值稳定。
def calc_woe_iv(df, bin_col, target=TARGET, smooth=0.5):
    """WOE = ln(坏账占比/好账占比)，每箱计数 +0.5 平滑避免除零；IV = Σ(坏占比-好占比)×WOE。"""
    g = df.groupby(bin_col, observed=True)[target].agg(['sum', 'count'])
    g.columns = ['bad', 'total']
    g['good'] = g['total'] - g['bad']
    bad_s = g['bad'] + smooth
    good_s = g['good'] + smooth
    g['bad_pct'] = bad_s / bad_s.sum()
    g['good_pct'] = good_s / good_s.sum()
    g['WOE'] = np.log(g['bad_pct'] / g['good_pct'])
    g['IV'] = (g['bad_pct'] - g['good_pct']) * g['WOE']
    return g, g['IV'].sum()


# ---------- 6.2 对每个候选特征计算 WOE 和 IV ----------
woe_map = {}
iv_map = {}
woe_detail = {}
for c in feature_cols:
    grp, iv = calc_woe_iv(train.assign(**{'__bin__': apply_bins(train[c], bins_map[c])}), '__bin__')
    woe_map[c] = grp['WOE'].to_dict()
    iv_map[c] = iv
    woe_detail[c] = grp

# ---------- 6.3 IV 排序与解读 ----------
iv_df = (pd.DataFrame({'特征': feature_cols, 'IV': [iv_map[c] for c in feature_cols]})
         .sort_values('IV', ascending=False).reset_index(drop=True))
print('IV 排序表（全部候选特征）：')
print(iv_df.round(4).to_string(index=False))
print('解读：grade(0.39)/int_rate(0.38) 是平台自身评级与定价，天然携带最强违约信息；'
      'loan_to_income(0.15)/term(0.14)/repay_burden(0.13) 反映杠杆与期限风险；'
      '底部 17 个特征 IV<0.02（多为大量为 0 的计数类、州/用途等），几乎无预测力，予以剔除。')
print()

# ---------- 6.4 IV 筛选（剔除 IV < 0.02 的弱预测力特征） ----------
# 为什么 IV<0.02 剔除：IV 衡量特征对目标的区分力，<0.02 几乎无预测力，入模只会增加噪声。
selected = [c for c in feature_cols if iv_map[c] >= 0.02]
dropped_iv = [c for c in feature_cols if iv_map[c] < 0.02]
print('IV < 0.02 剔除:', dropped_iv)
print('IV 保留特征数:', len(selected))
print()


# ---------- 6.5 WOE 编码辅助函数 ----------
def woe_encode(df_, c):
    return apply_bins(df_[c], bins_map[c]).map(woe_map[c]).astype(float).fillna(0.0)


# ---------- 6.6 共线性诊断与剔除（相关系数 > 0.9） ----------
# 为什么做共线性剔除：两个高度相关(|相关|>0.9)的特征会相互抵消、系数不稳定（甚至符号翻转），
# 保留 IV 更高者即可，既不损失信息又保证评分卡系数稳健。
X_train_woe = pd.DataFrame({c: woe_encode(train, c) for c in selected})
corr = X_train_woe.corr()
drop_coll = set()
for i in range(len(selected)):
    for j in range(i + 1, len(selected)):
        a, b = selected[i], selected[j]
        if abs(corr.loc[a, b]) > 0.9 and a not in drop_coll and b not in drop_coll:
            drop_coll.add(a if iv_map[a] < iv_map[b] else b)
if drop_coll:
    print('共线性剔除（|WOE相关|>0.9）:', sorted(drop_coll))
selected = [c for c in selected if c not in drop_coll]
print('最终入模特征数:', len(selected))
print('最终入模特征:', selected)
print()
# =============================================================================
# 步骤 ⑦：逻辑回归建模与评估
# 目的：使用 WOE 编码后的特征训练逻辑回归模型，评估模型区分能力
# 关键原则：仅使用训练集拟合模型，测试集仅用于评估
# =============================================================================
print('=' * 70)
print('⑦ 逻辑回归建模与评估')
print('=' * 70)

# ---------- 7.1 准备训练集和测试集的 WOE 特征矩阵 ----------
X_train = X_train_woe[selected]
X_test = pd.DataFrame({c: woe_encode(test, c) for c in selected})
y_train = train[TARGET].values
y_test = test[TARGET].values

# ---------- 7.2 训练逻辑回归模型 ----------
# 为什么喂 WOE 值而非原始值：WOE 编码后的特征与违约 log-odds 近似线性，逻辑回归能更好拟合，
# 系数也单调可解释（系数为正 → 该特征 WOE 越高、风险越高）。
# 为什么看 AUC/KS：AUC 衡量排序区分力，KS 衡量好坏样本分布的最大分离度，是评分卡的标准验收指标。
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

# ---------- 7.3 在测试集上预测并计算评估指标 ----------
prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, prob)
fpr, tpr, _ = roc_curve(y_test, prob)
ks = (tpr - fpr).max()
print('测试集 AUC = %.4f' % auc)
print('测试集 KS  = %.4f' % ks)
print()

# ---------- 7.4 绘制 ROC 曲线与 KS 曲线 ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(fpr, tpr, color='crimson', lw=2, label='AUC = %.3f' % auc)
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('假正率 FPR'); axes[0].set_ylabel('真正率 TPR')
axes[0].set_title('ROC 曲线'); axes[0].legend()
ks_idx = np.argmax(tpr - fpr)
axes[1].plot(tpr, label='TPR（累计坏账率）', color='crimson')
axes[1].plot(fpr, label='FPR（累计好账误判率）', color='steelblue')
axes[1].axvline(ks_idx, color='gray', ls='--', lw=1, label='KS = %.3f' % ks)
axes[1].set_xlabel('样本（按风险从高到低排序）'); axes[1].legend()
axes[1].set_title('KS 曲线')
plt.tight_layout()
plt.savefig('fig3_roc_ks.png', dpi=120)

# ---------- 7.5 输出模型系数（按绝对值排序） ----------
coef_df = pd.DataFrame({'特征': selected, '系数': model.coef_[0]})
coef_df = coef_df.sort_values('系数', key=abs, ascending=False).reset_index(drop=True)
print('逻辑回归系数（按 |系数| 排序）：')
print(coef_df.round(4).to_string(index=False))
print()


# =============================================================================
# 步骤 ⑧：评分映射与评分卡输出
# 目的：将逻辑回归系数转换为标准评分卡，便于业务人员使用
# =============================================================================
print('=' * 70)
print('⑧ 评分映射与评分卡')
print('=' * 70)

# ---------- 8.1 评分参数计算（600/50 惯例） ----------
# 为什么用 600/50 惯例：行业标准，好客户 odds=50(坏账率约2%)记 600 分，odds 翻倍 +50 分，分数直观可解释。
# Factor = 50/ln2 由「600 对应 odds=50、650 对应 odds=100」两个已知点联立解出。
p0, odds0, pdo = 600, 50, 50
factor = pdo / np.log(2)
offset = p0 - factor * np.log(odds0)
intercept = model.intercept_[0]
base_score = offset - factor * intercept
coef = dict(zip(selected, model.coef_[0]))

print('评分参数：Factor = %.4f | Offset = %.4f | 基准分(base_score) = %.1f' % (factor, offset, base_score))
print('（含义：好客户 odds=50 对应 600 分，odds 翻倍 +50 分）')
print()

# ---------- 8.2 生成评分卡明细表 ----------
scorecard_rows = []
for c in selected:
    grp = woe_detail[c]
    for bin_label, row in grp.iterrows():
        woe_val = row['WOE']
        # 为什么得分带负号：逻辑回归输出的是违约 log-odds(坏/好)，评分要用「好/坏」的 log-odds，
        # 二者差一个负号，故每档得分取负号（WOE 高=风险高=扣分）。
        score = -factor * coef[c] * woe_val
        scorecard_rows.append({
            '变量': c, '分箱': bin_label, '样本数': int(row['total']),
            'WOE': round(woe_val, 4), '得分': round(score, 1)
        })
scorecard = pd.DataFrame(scorecard_rows)

# 分箱标签美化：term 补「个月」，与数据字典一致（36/60 个月）
scorecard.loc[scorecard['变量'] == 'term', '分箱'] = \
    scorecard.loc[scorecard['变量'] == 'term', '分箱'].astype(str) + ' 个月'

# 变量按 IV 降序、箱内按得分降序
var_order = sorted(selected, key=lambda c: -iv_map[c])
scorecard['_o'] = scorecard['变量'].map({v: i for i, v in enumerate(var_order)})
scorecard = scorecard.sort_values(['_o', '得分'], ascending=[True, False]).drop(columns='_o').reset_index(drop=True)

# ---------- 8.3 格式化得分显示（正数加 + 号） ----------
# 得分显示：正数加 + 号（模板要求 +38.5 / -23.1 样式）
def fmt_score(s):
    return ('+' if s >= 0 else '') + f'{s:.1f}'
scorecard['得分'] = scorecard['得分'].apply(fmt_score)

# 基础分行插入表头（变量=基础分，其余用 — 占位）
base_row = pd.DataFrame([{'变量': '基础分', '分箱': '—', '样本数': '—', 'WOE': '—', '得分': f'{base_score:.1f}'}])
scorecard_table = pd.concat([base_row, scorecard], ignore_index=True)

print('表 1：标准评分卡（得分>0 加分、得分<0 减分）')
print(scorecard_table.to_string(index=False))
print('打分：总分 = 基础分 + 各变量命中档位得分之和，分越高风险越低。')
scorecard_table.to_csv('评分卡_表1.csv', index=False, encoding='utf-8-sig')
print()

# ---------- 8.4 打分函数与一致性校验 ----------
def score_customer(df_, bins_map_, woe_map_, coef_, factor_, base_score_):
    total = np.full(len(df_), base_score_)
    for c in selected:
        total -= factor_ * coef_[c] * apply_bins(df_[c], bins_map_[c]).map(woe_map_[c]).astype(float).fillna(0.0)
    return total

test = test.copy()
test['score'] = score_customer(test, bins_map, woe_map, coef, factor, base_score)

prob_test = model.predict_proba(X_test)[:, 1]
odds_good = (1 - prob_test) / prob_test
diff = np.abs(test['score'].values - (offset + factor * np.log(odds_good))).max()
print('评分一致性校验：评分卡求和 vs 概率反推，最大绝对差 = %.4f' % diff)
print('测试集分数分布：')
print(test['score'].describe().round(1).to_string())
print()

# ---------- 8.5 新客户打分演示 ----------
# 写清「如何给一个新客户打分」：取测试集第 1 位客户，演示 查档→取分→加总
demo = test.iloc[0]
print('如何给一个新客户打分（以测试集第 1 位客户为例）：')
print('  ① 看每个特征落在哪个档位 → ② 查评分卡取该档得分 → ③ 总分 = 基础分 + 各档得分之和。')
demo_rows = []
for c in selected:
    bl = apply_bins(pd.Series([demo[c]]), bins_map[c]).iloc[0]   # 原始档位标签，用于查 WOE
    w = woe_map[c].get(bl, 0.0)
    bl_disp = bl + ' 个月' if c == 'term' else bl                 # 展示时 term 补「个月」
    demo_rows.append({'特征': c, '命中档位': bl_disp, '得分': fmt_score(-factor * coef[c] * w)})
demo_df = pd.DataFrame(demo_rows)
print(demo_df.to_string(index=False))
print('  基础分 %.1f + 各档得分 = 该客户总分 %.1f（与 score_customer 结果一致）；再按表 2 阈值判定通过/复核/拒绝。'
      % (base_score, demo['score']))
print()


# =============================================================================
# 步骤 ⑨：风险分层、cut-off 校准与结论
# 目的：确定拒绝/复核/通过的分数阈值，输出最终放贷决策建议
# 关键原则：cut-off 阈值仅在训练集上确定，测试集仅用于评估效果
# =============================================================================
print('=' * 70)
print('⑨ 风险分层与 cut-off 校准')
print('=' * 70)

# ---------- 9.1 按分数分 10 档，验证区分度 ----------
# 为什么按 10 档看坏账率：验证分数是否单调区分风险（分数越高坏账率越低），是评分卡有效性的直接证据。
test['score_decile'] = pd.qcut(test['score'], 10, labels=False, duplicates='drop')
decile = test.groupby('score_decile', observed=True).agg(
    样本数=('score', 'count'), 最低分=('score', 'min'), 最高分=('score', 'max'), 坏账率=(TARGET, 'mean'))
decile['坏账率'] = (decile['坏账率'] * 100).round(2)
decile = decile.reset_index().rename(columns={'score_decile': '分数档'})
decile['分数档'] = decile['分数档'] + 1
print('10 档分数 → 坏账率（验证区分度，应单调递减）：')
print(decile.to_string(index=False))
print()

# ---------- 9.2 cut-off 阈值校准（仅在训练集上确定） ----------
# 为什么 cut-off 按本组人群分位校准：本数据坏账率 19.25%，教科书 600/650 对应约 2% 坏账率，
# 直接套用会把绝大多数客户拒掉；必须按本人群分数分布定阈值（这里以 30%/70% 分位切三档）。
# 为什么阈值在训练集上定：cut-off 阈值也是「用数据学出来的量」，只能由训练集决定（防泄漏），
# 测试集只用于评估（报告各档坏账率/拒单率）。
train['score'] = score_customer(train, bins_map, woe_map, coef, factor, base_score)
q_rej, q_app = train['score'].quantile([0.3, 0.7])
print('cut-off 校准：以「自动通过上限坏账率≈10%、强制拒绝下限≈25%」为业务目标，'
      '映射到本组人群 30%/70% 分位：')
print('  拒绝     < %.0f 分' % q_rej)
print('  人工复核  %.0f ~ %.0f 分' % (q_rej, q_app))
print('  通过     > %.0f 分' % q_app)

# ---------- 9.3 应用阈值到测试集，生成决策表 ----------
def decision(s):
    return '拒绝' if s < q_rej else ('人工复核' if s < q_app else '通过')

test['decision'] = test['score'].apply(decision)

# 表 2：按模板统计「分数段 / 样本占比 / 坏账率 / 建议动作」三档
def seg_stats(mask):
    share = mask.sum() / len(test) * 100
    br = test.loc[mask, TARGET].mean() * 100
    return share, br

segments = [
    ('通过',   test['score'] >= q_app, '≥ %.0f' % q_app),
    ('人工复核', (test['score'] >= q_rej) & (test['score'] < q_app), '%.0f ~ %.0f' % (q_rej, q_app)),
    ('拒绝',   test['score'] < q_rej, '< %.0f' % q_rej),
]
table2_rows = []
seg_br = {}    # 供画图用的数值坏账率（table2 里是格式化字符串，这里保留数值）
for action, mask, seg in segments:
    share, br = seg_stats(mask)
    seg_br[action] = br
    table2_rows.append({'分数段': seg, '样本占比': f'{share:.1f}%', '坏账率': f'{br:.2f}%', '建议动作': action})
table2 = pd.DataFrame(table2_rows)

rej_rate = (test['decision'] == '拒绝').mean() * 100
app_rate = (test['decision'] == '通过').mean() * 100
print()
print('表 2：分数 → 违约率映射与 cut-off')
print(table2.to_string(index=False))
print('拒单率: %.1f%% | 直接通过率: %.1f%% | 人工复核率: %.1f%%'
      % (rej_rate, app_rate, 100 - rej_rate - app_rate))
table2.to_csv('评分卡_表2_决策.csv', index=False, encoding='utf-8-sig')
print()

# ---------- 9.4 输出放贷决策建议 ----------
print('放贷决策建议：≥%.0f 分直接通过（坏账率 %.1f%%）、<%.0f 分直接拒绝（坏账率 %.1f%%）、'
      '中间人工复核；据此可自动处理约 %.0f%% 的申请（通过+拒绝），仅 %.0f%% 需人工介入，'
      '通过人群坏账率 %.1f%% 远低于整体 19.25%%，有效控制风险。'
      % (q_app, seg_br['通过'], q_rej, seg_br['拒绝'],
         app_rate + rej_rate, 100 - app_rate - rej_rate, seg_br['通过']))
print()

# ---------- 9.5 风险分层可视化 ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].bar(decile['分数档'].astype(str), decile['坏账率'], color='crimson')
axes[0].set_xlabel('分数档（1=最低分/最高风险 → 10=最高分/最低风险）')
axes[0].set_ylabel('坏账率 %')
axes[0].set_title('10 档分数 × 坏账率（区分度）')
for x, y in enumerate(decile['坏账率']):
    axes[0].text(x, y + 0.5, f'{y:.1f}', ha='center', fontsize=8)
colors = {'通过': 'seagreen', '人工复核': 'goldenrod', '拒绝': 'crimson'}
order = ['通过', '人工复核', '拒绝']
axes[1].bar(order, [seg_br[k] for k in order], color=[colors[k] for k in order])
axes[1].set_ylabel('坏账率 %')
axes[1].set_title('通过 / 人工复核 / 拒绝 三档坏账率')
for x, k in enumerate(order):
    axes[1].text(x, seg_br[k] + 0.5, f"{seg_br[k]:.2f}%", ha='center')
plt.tight_layout()
plt.savefig('fig4_risk_stratify.png', dpi=120)


# =============================================================================
# 附录 A：模型对比（RandomForest / XGBoost）
# 目的：验证可解释的 LR 模型是否足以胜任，若树模型 AUC 相当则无需牺牲解释性
# =============================================================================
print('=' * 70)
print('附录 A：模型对比（RandomForest / XGBoost）')
print('=' * 70)

# ---------- A.1 准备树模型训练数据（One-Hot 编码） ----------
def onehot_feature_matrix(df_):
    cats = [c for c in cat_cols if c in selected]
    X = df_[selected].copy()
    for c in cats:
        X[c] = X[c].astype(str)
    return pd.get_dummies(X, columns=cats)

X_tr_raw = onehot_feature_matrix(train)
X_te_raw = onehot_feature_matrix(test)
X_tr_raw, X_te_raw = X_tr_raw.align(X_te_raw, join='left', axis=1)
X_te_raw = X_te_raw.fillna(0)

# ---------- A.2 随机森林 ----------
# 为什么做模型对比：检验树模型是否显著优于 LR；若 AUC 相当，则证明「可解释的 LR」足以胜任，
# 不必牺牲解释性（评分卡必须能逐条向客户解释得分构成）。
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=50, n_jobs=-1, random_state=SEED)
rf.fit(X_tr_raw, y_train)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_te_raw)[:, 1])

# ---------- A.3 XGBoost / LightGBM / HistGradientBoosting 备选 ----------
# 优先用题目要求的 XGBoost；不可用则退到 LightGBM，再退到 sklearn HistGradientBoosting
try:
    import xgboost as xgb
    gbm = xgb.XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5,
                            min_child_weight=50, subsample=0.8, colsample_bytree=0.8,
                            eval_metric='auc', random_state=SEED)
    gbm.fit(X_tr_raw, y_train)
    auc_gbm = roc_auc_score(y_test, gbm.predict_proba(X_te_raw)[:, 1])
    gbm_name = 'XGBoost'
except Exception:
    try:
        import lightgbm as lgb
        gbm = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                                 min_child_samples=50, random_state=SEED, verbose=-1)
        gbm.fit(X_tr_raw, y_train)
        auc_gbm = roc_auc_score(y_test, gbm.predict_proba(X_te_raw)[:, 1])
        gbm_name = 'LightGBM'
    except Exception:
        from sklearn.ensemble import HistGradientBoostingClassifier
        gbm = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                             max_leaf_nodes=31, min_samples_leaf=50, random_state=SEED)
        gbm.fit(X_tr_raw, y_train)
        auc_gbm = roc_auc_score(y_test, gbm.predict_proba(X_te_raw)[:, 1])
        gbm_name = 'HistGradientBoosting'

# ---------- A.4 输出对比结果 ----------
print('模型对比（同一清洗/特征，70/30 划分）：')
print('  Logistic Regression(WOE)  AUC = %.4f  ← 主模型' % auc)
print('  RandomForest              AUC = %.4f' % auc_rf)
print('  %-26s AUC = %.4f' % (gbm_name, auc_gbm))
print('说明：树模型 AUC 与 LR 基本相当（略高或持平），但评分卡要求可解释、能逐条给客户'
      '解释得分构成，监管也偏好单调可解释模型，故仍以 LR 为主模型；树模型可作为辅助校验。')
print()


# =============================================================================
# 附录 B：时间切分验证（按 issue_month 前 80% 训练 / 后 20% 测试）
# 目的：模拟「用历史预测未来」，检验模型在样本外时间上的稳定性
# 关键原则：所有拟合参数（清洗、特征、分箱、WOE、LR）均在时间训练集上重新计算
# =============================================================================
print('=' * 70)
print('附录 B：时间切分验证（按 issue_month）')
print('=' * 70)

# ---------- B.1 按月份切分数据集 ----------
# 为什么做时间切分：随机切分可能让「未来信息」泄漏；按放款月份前 80% 训练/后 20% 测试，
# 完全模拟「用历史预测未来」，检验模型在样本外时间的稳定性。
df_raw2 = df_raw.copy()
df_raw2['ym'] = df_raw2['issue_month'].str.replace('-', '').astype(int)
df_raw2 = df_raw2.sort_values('ym').reset_index(drop=True)
n_ts = len(df_raw2)
ts_cut = int(n_ts * 0.8)
tr_raw_ts, te_raw_ts = df_raw2.iloc[:ts_cut], df_raw2.iloc[ts_cut:]
print('时间切分：训练 %d 行(%.2f) | 测试 %d 行(%.2f)'
      % (len(tr_raw_ts), len(tr_raw_ts) / n_ts, len(te_raw_ts), len(te_raw_ts) / n_ts))
print('  训练期 issue_month:', tr_raw_ts['issue_month'].min(), '~', tr_raw_ts['issue_month'].max())
print('  测试期 issue_month:', te_raw_ts['issue_month'].min(), '~', te_raw_ts['issue_month'].max())

# ---------- B.2 在时间训练集上重新执行全部拟合流程 ----------
# 同样 fit/apply：清洗、特征、分箱、WOE、LR 全部只在时间训练集拟合
cp_ts = clean_fit(tr_raw_ts)
tr_c_ts = clean_apply(tr_raw_ts, cp_ts)
te_c_ts = clean_apply(te_raw_ts, cp_ts)
fp_ts = feat_fit(tr_c_ts)
tr_ts = feat_apply(tr_c_ts, fp_ts)
te_ts = feat_apply(te_c_ts, fp_ts)

bins_ts = {c: build_bins(tr_ts[c], kind_map[c]) for c in feature_cols}
woe_ts = {}
iv_ts = {}
for c in feature_cols:
    grp, iv = calc_woe_iv(tr_ts.assign(**{'__bin__': apply_bins(tr_ts[c], bins_ts[c])}), '__bin__')
    woe_ts[c] = grp['WOE'].to_dict()
    iv_ts[c] = iv

# ---------- B.3 特征选择必须在时间训练集上重新做 ----------
# 关键：特征选择（IV 筛选 + 共线性剔除）必须在时间训练集上重新做，不能继承随机划分的 selected。
# 否则随机训练集里已混入后 20% 的样本，等于特征选择「看过未来」，造成特征选择信息污染。
def woe_encode_ts(df_, c):
    return apply_bins(df_[c], bins_ts[c]).map(woe_ts[c]).astype(float).fillna(0.0)

sel_iv_ts = [c for c in feature_cols if iv_ts[c] >= 0.02]
Xtr_ts_all = pd.DataFrame({c: woe_encode_ts(tr_ts, c) for c in sel_iv_ts})
corr_ts = Xtr_ts_all.corr()
drop_coll_ts = set()
for i in range(len(sel_iv_ts)):
    for j in range(i + 1, len(sel_iv_ts)):
        a, b = sel_iv_ts[i], sel_iv_ts[j]
        if abs(corr_ts.loc[a, b]) > 0.9 and a not in drop_coll_ts and b not in drop_coll_ts:
            drop_coll_ts.add(a if iv_ts[a] < iv_ts[b] else b)
sel_ts = [c for c in sel_iv_ts if c not in drop_coll_ts]
print('时间训练集重新 IV 筛选 + 共线性剔除 → 入模特征 %d 个（随机划分入模 %d 个）'
      % (len(sel_ts), len(selected)))
print('  特征集合是否一致：%s' % ('完全一致' if set(sel_ts) == set(selected) else '有差异 → ' + str(sorted(set(sel_ts) ^ set(selected)))))

# ---------- B.4 时间切分模型训练与评估 ----------
Xtr_ts = pd.DataFrame({c: woe_encode_ts(tr_ts, c) for c in sel_ts})
Xte_ts = pd.DataFrame({c: woe_encode_ts(te_ts, c) for c in sel_ts})
m_ts = LogisticRegression(max_iter=2000).fit(Xtr_ts, tr_ts[TARGET].values)
p_ts = m_ts.predict_proba(Xte_ts)[:, 1]
auc_ts = roc_auc_score(te_ts[TARGET].values, p_ts)
fpr_ts, tpr_ts, _ = roc_curve(te_ts[TARGET].values, p_ts)
ks_ts = (tpr_ts - fpr_ts).max()
print('时间切分（OOT 样本外）测试 AUC = %.4f | KS = %.4f' % (auc_ts, ks_ts))
print('对比随机划分 AUC = %.4f | KS = %.4f；二者接近，说明模型时间上稳定（无过拟合/泄漏）。' % (auc, ks))
print()


# =============================================================================
# 附录 C：衍生特征对模型表现的提升
# 目的：量化新特征的边际贡献，证明特征工程确实提升了模型表现
# =============================================================================
print('=' * 70)
print('附录 C：衍生特征对模型表现的提升')
print('=' * 70)

# ---------- C.1 对比有无衍生特征时的 AUC ----------
# 为什么对比有无衍生特征：量化新特征的边际贡献，证明特征工程确实提升了模型表现。
base_features = [c for c in selected if c not in DERIVED]
X_tr_base = X_train[base_features]
X_te_base = X_test[base_features]
m_base = LogisticRegression(max_iter=2000).fit(X_tr_base, y_train)
auc_base = roc_auc_score(y_test, m_base.predict_proba(X_te_base)[:, 1])

# ---------- C.2 输出提升结果 ----------
print('不含衍生特征 LR AUC = %.4f' % auc_base)
print('含衍生特征 LR AUC   = %.4f（提升 %.4f）' % (auc, auc - auc_base))
print('说明：loan_to_income 的 IV=%.4f 排名第 3，本身有强区分力；但因与 dti/annual_inc/loan_amnt'
      ' 信息重叠，入模后整体 AUC 提升有'
      '限。衍生特征的价值更多在于业务可解释性'
      '（相对负担/杠杆比绝对金额更可比）。' % iv_map['loan_to_income'])


# =============================================================================
# PPT 图表生成
# =============================================================================
print('=' * 70)
print('PPT 图表生成')
print('=' * 70)

from matplotlib.colors import LinearSegmentedColormap, Normalize

# ---------- 配色方案 ----------
# 顺序色带（蓝 浅→深，编码「量级/大小」）；发散色带（蓝→灰→红，编码「正负/极性」）
cmap_blue = LinearSegmentedColormap.from_list('blue_ramp', ['#cde2fb', '#86b6ef', '#3987e5', '#184f95', '#0d366b'])
cmap_div = LinearSegmentedColormap.from_list('div', ['#184f95', '#86b6ef', '#f0efec', '#ec835a', '#1baf7a', '#e34948'])
INK = '#0b0b0b'
MUTED = '#898781'
# -----------------------------------------------------------------------------
# 1. 导入所需库与全局配置
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # 非交互后端：plt.show() 不阻塞，图仅存盘
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体，防止图表中文乱码
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve

# 设置 Pandas 显示选项，便于查看宽表
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# 固定随机种子（全项目统一）
SEED = 42
np.random.seed(SEED)
TARGET = 'y'  # 目标变量名称
# =============================================================================
# 步骤 ①：业务理解与数据探索（EDA）
# 目的：全面了解数据分布、缺失情况、异常值及关键变量与违约率的关系
# =============================================================================
print('=' * 70)
print('① 业务理解与数据探索（EDA）')
print('=' * 70)

# ---------- 1.1 加载数据与目标变量概览 ----------
df_raw = pd.read_csv('lc_loan_data.csv')
print('数据形状:', df_raw.shape)
print('目标变量 y：1=违约(坏) 0=履约(好)')
print(df_raw[TARGET].value_counts().rename({0: '好客户', 1: '坏客户'}).to_string())
print('坏账率: %.2f%%' % (df_raw[TARGET].mean() * 100))
print()

# ---------- 1.2 全量缺失值诊断（含缺失为0的列） ----------
# 目的：明确每列缺失率，为后续缺失处理提供依据
miss = df_raw.isna().sum()
miss_table = (miss.rename('缺失数').to_frame()
              .assign(**{'缺失率%': lambda x: (x['缺失数'] / len(df_raw) * 100).round(2)}))
print('缺失值一览（全部 %d 列的缺失率，含缺失为 0 的列）：' % len(miss_table))
print(miss_table.to_string())
print()

# ---------- 1.3 数值型变量描述统计 ----------
# 为什么补全量：单变量分布不能只看少数几个变量，应对所有候选数值变量给描述统计，
# 便于全面发现极端值/偏态/量纲，为后续清洗与分箱提供依据。
EDA_NUMERIC = ['loan_amnt', 'funded_amnt', 'int_rate', 'installment', 'emp_length',
               'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq',
               'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
               'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal',
               'total_rev_hi_lim', 'mths_since_earliest_cr_line']
print('单变量分布（全量数值变量 describe）：')
print(df_raw[EDA_NUMERIC].describe().round(2).to_string())
print()

# ---------- 1.4 类别型变量频数统计 ----------
EDA_CAT = ['grade', 'term', 'home_ownership', 'verification_status',
           'purpose', 'initial_list_status', 'addr_state', 'sub_grade']
print('单变量分布（类别变量频数统计，含各档样本数）：')
for c in EDA_CAT:
    vc = df_raw[c].astype(str).value_counts()
    top = vc.head(12)
    line = ', '.join(f'{k}={v}' for k, v in top.items())
    if len(vc) > 12:
        line += f', ...(共{len(vc)}类)'
    print(f'  {c}: {line}')
print()

# ---------- 1.5 类别变量频数条形图（用于PPT展示） ----------
# 只画核心入模类别变量，直观看出各档分布与稀有类
cat_plot = ['grade', 'term', 'home_ownership', 'verification_status', 'purpose', 'initial_list_status']
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), cat_plot):
    vc = df_raw[c].astype(str).value_counts()
    if len(vc) > 10:            # 高基数变量（如 purpose 14 类）只画前 10 类，避免柱子过密
        vc = vc.head(10)
    sns.barplot(x=vc.index, y=vc.values, ax=ax, color='steelblue')
    ax.set_title(c, fontsize=12)
    ax.set(ylabel='样本数', xlabel=None)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
plt.suptitle('类别变量频数分布', fontsize=14)
plt.tight_layout()
plt.savefig('fig_categorical.png', dpi=120)

# ---------- 1.6 重点异常值识别（基于数据字典和业务常识） ----------
print('异常值识别：')
print('  revol_util > 100% 的样本数:', (df_raw['revol_util'] > 100).sum())
print('  annual_inc 极端大值（99% 分位以上）:', (df_raw['annual_inc'] > df_raw['annual_inc'].quantile(0.99)).sum(),
      '，最大:', df_raw['annual_inc'].max())
print('  tot_coll_amt / tot_cur_bal / total_rev_hi_lim 三者同时缺失:',
      df_raw[['tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim']].isna().all(axis=1).sum())
print()

# ---------- 1.7 关键变量与违约率的初步关系（验证业务直觉） ----------
print('关键变量 × 违约率（初步关系）：')
print('  term 36个月 坏账率: %.2f%% | term 60个月 坏账率: %.2f%%'
      % (df_raw.loc[df_raw['term'] == 36, TARGET].mean() * 100,
         df_raw.loc[df_raw['term'] == 60, TARGET].mean() * 100))
grade_br = df_raw.groupby('grade', observed=True)[TARGET].agg(['mean', 'count'])
grade_br['mean'] = (grade_br['mean'] * 100).round(2)
print('  grade × 坏账率：')
print(grade_br.rename(columns={'mean': '坏账率%', 'count': '样本数'}).to_string())
print('  int_rate 与 grade 相关系数（有序映射后）:',
      round(df_raw['int_rate'].corr(df_raw['grade'].map({g: i for i, g in enumerate(sorted(df_raw['grade'].unique()))})), 4))
print()

# ---------- 1.8 清洗前箱线图（可视化极端值分布） ----------
_num_cols = ['loan_amnt', 'installment', 'annual_inc', 'dti', 'revol_bal',
             'revol_util', 'total_acc', 'mths_since_earliest_cr_line']
fig, axes = plt.subplots(2, 4, figsize=(18, 6))
for ax, c in zip(axes.ravel(), _num_cols):
    sns.boxplot(y=df_raw[c], ax=ax)
    ax.set_title(c, fontsize=10)
    ax.set(ylabel=None)
plt.suptitle('清洗前箱线图（可见 annual_inc / revol_util / revol_bal 存在极端值）', fontsize=13)
plt.tight_layout()
plt.savefig('fig1_eda_boxplot_before.png', dpi=120)
# ============================================================
# ② 数据清洗（fit/apply 分离：中位数与截尾阈值只在训练集 fit）
# ============================================================
print('=' * 70)
print('② 数据清洗')
print('=' * 70)


def clean_fit(df_train):
    """在训练集上学习填充中位数与截尾阈值，返回参数。

    为什么只在训练集 fit：中位数/分位点是用数据「学」出来的量，只能在训练集计算，
    测试集只套用。
    """
    return {
        'revol_util_med': df_train['revol_util'].median(),
        'tot_coll_amt_med': df_train['tot_coll_amt'].median(),
        'tot_cur_bal_med': df_train['tot_cur_bal'].median(),
        'total_rev_hi_lim_med': df_train['total_rev_hi_lim'].median(),
        'inc_cap': df_train['annual_inc'].quantile(0.99),
    }


def clean_apply(df_in, p):
    """套用训练集学到的参数做填充与截尾。"""
    df = df_in.copy()

    # ── 缺失值处理 ──
    # 为什么不用 dropna 扔样本：缺失本身有信息（如「从未逾期」的客户才缺 mths_since_last_delinq），
    # 直接丢弃会损失大量样本并引入偏差，所以改用「有业务含义的填充」保留信息。
    # 为什么金额类用中位数：金额变量右偏严重、存在极端值，中位数对极端值稳健、
    # 更能代表「典型客户」水平，均值会被少数巨款客户拉高。
    df['mths_since_last_delinq'] = df['mths_since_last_delinq'].fillna(-1)  # 56%缺失=从未逾期，用 -1 单独标记，分箱时自成「从未逾期」档
    df['revol_util'] = df['revol_util'].fillna(p['revol_util_med'])          # 32个缺失(0.06%)，金额类中位数填充
    df['collections_12_mths_ex_med'] = df['collections_12_mths_ex_med'].fillna(0)  # 99.5%为0，缺失大概率也是无催收，填 0
    df['tot_coll_amt'] = df['tot_coll_amt'].fillna(p['tot_coll_amt_med'])    # 金额类，中位数填充
    df['tot_cur_bal'] = df['tot_cur_bal'].fillna(p['tot_cur_bal_med'])        # 金额类，中位数填充
    df['total_rev_hi_lim'] = df['total_rev_hi_lim'].fillna(p['total_rev_hi_lim_med'])  # 金额类，中位数填充（三者同批缺失，同一批无余额信息客户）

    # ── 异常值处理 ──
    # 为什么 revol_util 截尾到 100：循环额度使用率按定义上限就是 100%，>100% 属于口径/录入异常，
    # 截到 100 更符合业务含义，且比删除更保守（只改值、不丢样本）。
    df['revol_util'] = df['revol_util'].clip(upper=100)
    # 为什么极端收入用 winsorize(99%分位截尾) 而非删除：收入存在数百万美元极端值，
    # 删除会损失样本且带主观性；winsorize 到 99% 分位既保留样本、又抑制极端值对分箱/WOE 的扰动。
    df['annual_inc'] = df['annual_inc'].clip(upper=p['inc_cap'])
    return df


# ============================================================
# ③ 特征工程（衍生特征 ≥3；同样 fit/apply 分离）
# ============================================================
print('=' * 70)
print('③ 特征工程（衍生特征）')
print('=' * 70)

# 为什么构造比值特征：绝对金额（贷款额/循环欠款）在不同收入人群间不可比，
# 除以年收入后变成「相对负担/杠杆」，既消除量纲差异、业务含义也更清晰（还贷压力、杠杆水平）。
DERIVED = ['repay_burden', 'loan_to_income', 'revol_bal_to_income',
           'tot_bal_to_income', 'credit_line_util', 'installment_to_loan',
           'delinq_flag', 'inq_flag', 'pub_rec_flag', 'high_util_flag']


def _derive(df):
    inc = df['annual_inc'].replace(0, np.nan)
    df['repay_burden'] = df['installment'] * 12 / inc         # 年还款负担率
    df['loan_to_income'] = df['loan_amnt'] / inc               # 贷款额收入比
    df['revol_bal_to_income'] = df['revol_bal'] / inc          # 循环负债收入比
    df['tot_bal_to_income'] = df['tot_cur_bal'] / inc          # 总余额收入比
    df['credit_line_util'] = df['revol_bal'] / df['total_rev_hi_lim'].replace(0, np.nan)  # 循环额度使用率
    df['installment_to_loan'] = df['installment'] * 12 / df['loan_amnt'].replace(0, np.nan)  # 年度还款额相对本金
    df['delinq_flag'] = (df['delinq_2yrs'] > 0).astype(int)  # 近两年是否发生过逾期
    df['inq_flag'] = (df['inq_last_6mths'] > 0).astype(int)  # 近半年是否有信用查询
    df['pub_rec_flag'] = (df['pub_rec'] > 0).astype(int)  # 是否存在公共不良记录
    df['high_util_flag'] = (df['revol_util'] >= 80).astype(int)  # 是否处于较高额度使用状态
    return df


def feat_fit(df_train):
    """在训练集上计算衍生特征的截尾/填充参数。"""
    tmp = _derive(df_train.copy())
    p = {}
    for c in DERIVED:
        s = tmp[c].replace([np.inf, -np.inf], np.nan)
        p[f'{c}_cap'] = s.quantile(0.99)
        p[f'{c}_med'] = s.median()
    return p


def feat_apply(df_in, p):
    """构造衍生特征，并用训练集参数做极端值处理。"""
    df = _derive(df_in.copy())
    for c in DERIVED:
        df[c] = df[c].replace([np.inf, -np.inf], np.nan)
        df[c] = df[c].clip(upper=p[f'{c}_cap'])
        df[c] = df[c].fillna(p[f'{c}_med'])
    return df


print('衍生特征（5 个，≥3）及业务含义（前 3 个为题目建议方向，后 2 个为自行探索）：')
print('  repay_burden       年还款负担率 = 月供×12/年收入，越高还贷压力越大、越易违约')
print('  loan_to_income     贷款额收入比 = 申请金额/年收入，衡量贷款杠杆，越高越激进')
print('  revol_bal_to_income 循环负债收入比 = 循环欠款/年收入，存量循环负债的相对负担')
print('  tot_bal_to_income  总余额收入比 = 账户总余额/年收入，总负债规模相对收入')
print('  credit_line_util   循环额度使用率 = 循环欠款/授信总额度（交叉特征：用余额与额度两列重构使用率，与平台 revol_util 互为印证）')
print()

# ============================================================
# ④ 数据集划分（70/30，固定 seed）—— 先划分，再拟合清洗/特征参数
# ============================================================
print('=' * 70)
print('④ 数据集划分（70/30，固定 seed=42）')
print('=' * 70)

# 为什么 70/30 + stratify + 固定 seed：70% 保证模型学得充分、30% 保证评估可靠；
# stratify 让训练/测试坏账率一致（本数据 19.25%），避免抽样偏斜；固定 seed 保证可复现。
# 为什么先划分再清洗：填充中位数/截尾阈值/分箱边界等「用数据学出来的量」都只能在训练集计算，
# 测试集只套用，否则等于把测试集信息泄漏进模型（防泄漏是评分卡第一纪律）。
train_raw, test_raw = train_test_split(df_raw, test_size=0.3, stratify=df_raw[TARGET], random_state=SEED)

# 清洗：只在训练集 fit，训练/测试统一 apply
clean_p = clean_fit(train_raw)
train_c = clean_apply(train_raw, clean_p)
test_c = clean_apply(test_raw, clean_p)

# 特征工程：只在训练集 fit，训练/测试统一 apply
feat_p = feat_fit(train_c)
train = feat_apply(train_c, feat_p)
test = feat_apply(test_c, feat_p)

print(f'训练集 {len(train)} 行 | 测试集 {len(test)} 行')
print('训练集坏账率 %.2f%% | 测试集坏账率 %.2f%%' % (train[TARGET].mean() * 100, test[TARGET].mean() * 100))

# ── 缺失值处理审计──
# 原则：每个缺失字段都必须明确归属三类之一——①单独缺失/特殊箱 ②业务含义箱 ③训练集统计量填充；
#       训练/测试统一用训练集学到的规则，保证一致（防泄漏）。
print('缺失值处理策略（每列明确归属）：')
for col, strat in [
    ('mths_since_last_delinq', '填 -1 → 分箱时 -1 单独成「从未逾期」档（单独缺失/特殊箱）'),
    ('collections_12_mths_ex_med', '填 0 → 0 单独成箱（业务含义：缺失视为无催收）'),
    ('revol_util', '训练集中位数填充（训练集统计量）'),
    ('tot_coll_amt / tot_cur_bal / total_rev_hi_lim', '训练集中位数填充（训练集统计量，三者同批缺失）'),
]:
    print(f'  {col}: {strat}')
# 断言闭环：清洗+特征工程后训练/测试不得再有任何 NaN（否则 apply_bins 会把 NaN 误当字符串 "nan"）
assert train.isna().sum().sum() == 0 and test.isna().sum().sum() == 0, '清洗后仍存在缺失值！'
print('审计通过：清洗+特征工程后，训练集与测试集均无缺失值（NaN=0），缺失已按上述策略显式处理、训练/测试一致。')
print('  annual_inc 截尾阈值(训练集): %.0f' % clean_p['inc_cap'])
print()

fig, axes = plt.subplots(2, 4, figsize=(18, 6))
for ax, c in zip(axes.ravel(), _num_cols):
    sns.boxplot(y=train[c], ax=ax)
    ax.set_title(c, fontsize=10)
    ax.set(ylabel=None)
plt.suptitle('清洗后箱线图（训练集，极端值已处理）', fontsize=13)
plt.tight_layout()
plt.savefig('fig2_eda_boxplot_after.png', dpi=120)

# ============================================================
# ⑤ 分箱（训练集定边界，测试集套用）
# ============================================================
print('=' * 70)
print('⑤ 分箱')
print('=' * 70)

# 特征池：剔除 funded_amnt(与loan_amnt高度相关)、grade（由更细的 sub_grade 替代）、issue_month（时间不直接入模）；
#         sub_grade 用于保留平台评级的细粒度层级信息，int_rate 保留作为独立定价变量
# 策略4：用更细粒度的 sub_grade 替代 grade 入模。sub_grade 保留 A1~G5 的层级风险信息，
# 比仅使用 A~G 七档 grade 提供更细的风险排序；grade 仍用于 EDA 展示，但不与 sub_grade 同时入模。
cat_cols = ['term', 'sub_grade', 'home_ownership', 'verification_status',
            'purpose', 'addr_state', 'initial_list_status', 'delinq_flag', 'inq_flag', 'pub_rec_flag', 'high_util_flag']
cont_cols = ['loan_amnt', 'installment', 'emp_length', 'annual_inc', 'dti',
             'open_acc', 'revol_bal', 'revol_util', 'total_acc',
             'mths_since_earliest_cr_line', 'int_rate', 'installment_to_loan'] + [c for c in DERIVED if c not in ['installment_to_loan', 'delinq_flag', 'inq_flag', 'pub_rec_flag', 'high_util_flag']]
zero_cols = ['delinq_2yrs', 'inq_last_6mths', 'pub_rec', 'acc_now_delinq',
             'collections_12_mths_ex_med', 'tot_coll_amt']
special_cols = ['mths_since_last_delinq']  # -1 = 从未逾期

feature_cols = cat_cols + cont_cols + zero_cols + special_cols
print('候选特征数:', len(feature_cols))


# 为什么分箱：把连续变量切成有序区间，捕捉「不同区间对违约率的不同影响」（非线性），且对异常值稳健；
# WOE 和评分卡必须建立在离散档位上。
# 为什么等频分箱（而非等宽）：保证每箱样本数大致相等，避免某箱样本过少导致 WOE 不稳定。
def _q_edges(s, n_bins):
    qs = s.quantile([i / n_bins for i in range(1, n_bins)])
    return sorted(set(np.round(qs.values, 6)))


def build_bins(train_s, kind, n_bins=5, min_freq=0.02):
    """返回分箱规则。kind ∈ {'cont','zero','special','cat'}。"""
    if kind == 'cont':
        return {'edges': sorted(set([-np.inf] + _q_edges(train_s, n_bins) + [np.inf])), 'cat_map': None}
    if kind == 'zero':  # 为什么 0 单独成箱：这些变量绝大多数为 0，「0」是有业务含义的状态（无逾期/无查询），须与正数分开
        pos = train_s[train_s > 0]
        return {'edges': sorted(set([-np.inf, 0.0] + _q_edges(pos, max(n_bins - 1, 2)) + [np.inf])), 'cat_map': None}
    if kind == 'special':  # 特殊值(-1=从未逾期)单独成箱
        rest = train_s[train_s >= 0]
        return {'edges': sorted(set([-np.inf, -0.5] + _q_edges(rest, n_bins) + [np.inf])), 'cat_map': None}
    if kind == 'cat':  # 为什么稀有类合并：样本太少的类别 WOE 估计不可靠（方差大），合并可避免过拟合
        s = train_s.astype(str)
        vc = s.value_counts()
        keep = set(vc[vc / len(s) >= min_freq].index)
        return {'edges': None, 'cat_map': {v: (v if v in keep else '其他') for v in vc.index}}


def apply_bins(s, rule):
    """把分箱规则套到 Series 上，返回类别标签（字符串）。

    缺失值约定：缺失已在清洗阶段全部显式处理（见「缺失值处理审计」的 assert），
    故这里假定输入无 NaN；若真出现 NaN，说明上游清洗有遗漏，应回查而非静默吞掉。
    """
    if rule['cat_map'] is not None:
        return s.astype(str).map(rule['cat_map']).fillna('其他')
    return pd.cut(s, rule['edges'], include_lowest=True).astype(str)


kind_map = {}
for c in cat_cols:
    kind_map[c] = 'cat'
for c in cont_cols:
    kind_map[c] = 'cont'
for c in zero_cols:
    kind_map[c] = 'zero'
for c in special_cols:
    kind_map[c] = 'special'

NO_MERGE = {'grade'}  # 有序且有业务含义的小基数变量，不合并稀有类
bins_map = {c: build_bins(train[c], kind_map[c], min_freq=(0.0 if c in NO_MERGE else 0.02))
            for c in feature_cols}
print('分箱规则已在训练集上建立。示例：')
print('  term:', bins_map['term']['cat_map'])
print('  delinq_2yrs(0单独成箱) 边界:', bins_map['delinq_2yrs']['edges'])
print('  mths_since_last_delinq(-1单独成箱) 边界:', bins_map['mths_since_last_delinq']['edges'])
print()

# =============================================================================
# 步骤 ⑥：WOE 与 IV（证据权重与信息价值）
# 目的：将原始特征转换为 WOE 编码，通过 IV 筛选有效特征，并进行共线性诊断
# 关键原则：所有 WOE/IV 计算仅在训练集上进行，测试集仅使用 transform
# =============================================================================
print('=' * 70)
print('⑥ WOE 与 IV')
print('=' * 70)


# ---------- 6.1 WOE/IV 计算函数定义 ----------
# 为什么用 WOE：把原始取值换成「该档位的风险强度」，天然处理非线性/单调性，逻辑回归系数也有业务含义。
# 为什么 +0.5 平滑：某档若坏样本数为 0，取对数会除零/无穷，加平滑项保证数值稳定。
def calc_woe_iv(df, bin_col, target=TARGET, smooth=0.5):
    """WOE = ln(坏账占比/好账占比)，每箱计数 +0.5 平滑避免除零；IV = Σ(坏占比-好占比)×WOE。"""
    g = df.groupby(bin_col, observed=True)[target].agg(['sum', 'count'])
    g.columns = ['bad', 'total']
    g['good'] = g['total'] - g['bad']
    bad_s = g['bad'] + smooth
    good_s = g['good'] + smooth
    g['bad_pct'] = bad_s / bad_s.sum()
    g['good_pct'] = good_s / good_s.sum()
    g['WOE'] = np.log(g['bad_pct'] / g['good_pct'])
    g['IV'] = (g['bad_pct'] - g['good_pct']) * g['WOE']
    return g, g['IV'].sum()


# ---------- 6.2 对每个候选特征计算 WOE 和 IV ----------
woe_map = {}
iv_map = {}
woe_detail = {}
for c in feature_cols:
    grp, iv = calc_woe_iv(train.assign(**{'__bin__': apply_bins(train[c], bins_map[c])}), '__bin__')
    woe_map[c] = grp['WOE'].to_dict()
    iv_map[c] = iv
    woe_detail[c] = grp

# ---------- 6.3 IV 排序与解读 ----------
iv_df = (pd.DataFrame({'特征': feature_cols, 'IV': [iv_map[c] for c in feature_cols]})
         .sort_values('IV', ascending=False).reset_index(drop=True))
print('IV 排序表（全部候选特征）：')
print(iv_df.round(4).to_string(index=False))
print('解读：sub_grade 保留平台评级的细粒度风险信息；同时通过 WOE 相关性和 VIF 控制变量冗余，避免仅凭 IV 高低机械入模。')
print()

# ---------- 6.4 策略8：IV 阈值敏感性 + WOE 相关性 + VIF ----------
# IV 阈值不直接固定为某一个经验值，而是在训练集内部进行敏感性比较。
# 特别保留题目要求的 0.02 作为基准阈值，同时考察更宽/更严的候选阈值。
IV_THRESHOLDS = [0.005, 0.01, 0.02, 0.03, 0.05]
CORR_THRESHOLD = 0.90
VIF_THRESHOLD = 5.0

# 训练集 WOE 矩阵：所有 WOE 参数均来自训练集，测试集不会参与变量筛选。
# 先定义 WOE 编码函数，后续阈值敏感性、相关性、VIF 共用。
def woe_encode(df_, c):
    return apply_bins(df_[c], bins_map[c]).map(woe_map[c]).astype(float).fillna(0.0)

X_train_woe_all = pd.DataFrame({c: woe_encode(train, c) for c in feature_cols})


def reduce_by_woe_corr(features, threshold=CORR_THRESHOLD):
    """按 |WOE相关系数| 删除冗余变量，优先保留 IV 更高者。"""
    if len(features) <= 1:
        return list(features), []
    corr = X_train_woe_all[features].corr()
    drop = set()
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            a, b = features[i], features[j]
            if a in drop or b in drop:
                continue
            r = corr.loc[a, b]
            if pd.notna(r) and abs(r) > threshold:
                drop.add(a if iv_map[a] < iv_map[b] else b)
    return [c for c in features if c not in drop], sorted(drop)


def reduce_by_vif(features, threshold=VIF_THRESHOLD):
    """迭代删除 VIF 最高且超过阈值的变量，避免多重共线性导致系数不稳定。"""
    try:
        from statsmodels.stats.outliers_influence import variance_inflation_factor
    except ImportError:
        print('提示：未安装 statsmodels，跳过 VIF；将仅使用 WOE 相关性控制共线性。')
        return list(features), []

    current = list(features)
    removed = []
    while len(current) >= 2:
        X = X_train_woe_all[current].astype(float).copy()
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        # 常数列无法计算 VIF，直接移除。
        constant = [c for c in current if X[c].nunique(dropna=False) <= 1]
        if constant:
            current = [c for c in current if c not in constant]
            removed.extend(constant)
            continue
        arr = X.values
        vif_values = []
        for i in range(arr.shape[1]):
            try:
                vif_values.append(variance_inflation_factor(arr, i))
            except Exception:
                vif_values.append(np.inf)
        vif_series = pd.Series(vif_values, index=current)
        max_feature = vif_series.idxmax()
        max_vif = vif_series.max()
        if not np.isfinite(max_vif) or max_vif > threshold:
            current.remove(max_feature)
            removed.append(max_feature)
        else:
            break
    return current, removed


def select_features_by_threshold(iv_threshold):
    """给定 IV 阈值后，依次执行 IV → WOE相关性 → VIF。"""
    iv_selected = [c for c in feature_cols if iv_map[c] >= iv_threshold]
    after_corr, corr_drop = reduce_by_woe_corr(iv_selected, CORR_THRESHOLD)
    after_vif, vif_drop = reduce_by_vif(after_corr, VIF_THRESHOLD)
    return iv_selected, after_corr, after_vif, corr_drop, vif_drop

threshold_rows = []
threshold_details = {}
for threshold in IV_THRESHOLDS:
    iv_selected, corr_selected, final_selected, corr_drop, vif_drop = select_features_by_threshold(threshold)
    # 仅在训练集上做 5 折 CV，用于选择阈值；最终测试集完全不参与该选择。
    X_cv = X_train_woe_all[final_selected]
    cv_auc_mean = np.nan
    if len(final_selected) > 0:
        cv_model = LogisticRegression(max_iter=2000, C=1.0)
        cv_auc_mean = cross_val_score(
            cv_model, X_cv, train[TARGET].values,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
            scoring='roc_auc', n_jobs=-1
        ).mean()
    threshold_rows.append({
        'IV阈值': threshold,
        'IV后特征数': len(iv_selected),
        '相关性剔除数': len(corr_drop),
        'VIF剔除数': len(vif_drop),
        '最终特征数': len(final_selected),
        '训练集5折CV_AUC': cv_auc_mean
    })
    threshold_details[threshold] = {
        'iv_selected': iv_selected,
        'corr_selected': corr_selected,
        'final_selected': final_selected,
        'corr_drop': corr_drop,
        'vif_drop': vif_drop
    }

threshold_df = pd.DataFrame(threshold_rows)
print('IV 阈值敏感性分析（全部基于训练集）：')
print(threshold_df.round(4).to_string(index=False))

# 以训练集 CV AUC 为主选择阈值；0.02 明确纳入比较，不强制指定最终阈值。
best_threshold = float(threshold_df.loc[threshold_df['训练集5折CV_AUC'].idxmax(), 'IV阈值'])
selected = threshold_details[best_threshold]['final_selected']
print('\n选择结果：')
print('  最优 IV 阈值 = %.3f（候选阈值包含 0.020）' % best_threshold)
print('  IV 筛选后特征:', threshold_details[best_threshold]['iv_selected'])
print('  WOE相关性剔除:', threshold_details[best_threshold]['corr_drop'])
print('  VIF剔除:', threshold_details[best_threshold]['vif_drop'])
print('  最终入模特征数:', len(selected))
print('  最终入模特征:', selected)
print()

# 输出最终 WOE 矩阵，供后续逻辑回归使用。
X_train_woe = X_train_woe_all[selected]

# =============================================================================
# 步骤 ⑦：逻辑回归建模与评估
# 目的：使用 WOE 编码后的特征训练逻辑回归模型，评估模型区分能力
# 关键原则：仅使用训练集拟合模型，测试集仅用于评估
# =============================================================================
print('=' * 70)
print('⑦ 逻辑回归建模与评估')
print('=' * 70)

# ---------- 7.1 准备训练集和测试集的 WOE 特征矩阵 ----------
X_train = X_train_woe[selected]
X_test = pd.DataFrame({c: woe_encode(test, c) for c in selected})
y_train = train[TARGET].values
y_test = test[TARGET].values

# ---------- 7.2 训练逻辑回归模型 ----------
# 为什么喂 WOE 值而非原始值：WOE 编码后的特征与违约 log-odds 近似线性，逻辑回归能更好拟合，
# 系数也单调可解释（系数为正 → 该特征 WOE 越高、风险越高）。
# 为什么看 AUC/KS：AUC 衡量排序区分力，KS 衡量好坏样本分布的最大分离度，是评分卡的标准验收指标。
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
C_grid = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
cv_auc = []
for c_val in C_grid:
    m = LogisticRegression(max_iter=2000, C=c_val)
    cv_auc.append(cross_val_score(m, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1).mean())
best_C = C_grid[int(np.argmax(cv_auc))]
print('训练集5折CV选择 LogisticRegression C = %.2f（CV AUC=%.4f）' % (best_C, max(cv_auc)))
model = LogisticRegression(max_iter=2000, C=best_C)
model.fit(X_train, y_train)

# ---------- 7.3 在测试集上预测并计算评估指标 ----------
prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, prob)
fpr, tpr, _ = roc_curve(y_test, prob)
ks = (tpr - fpr).max()
print('测试集 AUC = %.4f' % auc)
print('测试集 KS  = %.4f' % ks)
print()

# ---------- 7.4 绘制 ROC 曲线与 KS 曲线 ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(fpr, tpr, color='crimson', lw=2, label='AUC = %.3f' % auc)
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('假正率 FPR'); axes[0].set_ylabel('真正率 TPR')
axes[0].set_title('ROC 曲线'); axes[0].legend()
ks_idx = np.argmax(tpr - fpr)
axes[1].plot(tpr, label='TPR（累计坏账率）', color='crimson')
axes[1].plot(fpr, label='FPR（累计好账误判率）', color='steelblue')
axes[1].axvline(ks_idx, color='gray', ls='--', lw=1, label='KS = %.3f' % ks)
axes[1].set_xlabel('样本（按风险从高到低排序）'); axes[1].legend()
axes[1].set_title('KS 曲线')
plt.tight_layout()
plt.savefig('fig3_roc_ks.png', dpi=120)

# ---------- 7.5 输出模型系数（按绝对值排序） ----------
coef_df = pd.DataFrame({'特征': selected, '系数': model.coef_[0]})
coef_df = coef_df.sort_values('系数', key=abs, ascending=False).reset_index(drop=True)
print('逻辑回归系数（按 |系数| 排序）：')
print(coef_df.round(4).to_string(index=False))
print()


# =============================================================================
# 步骤 ⑧：评分映射与评分卡输出
# 目的：将逻辑回归系数转换为标准评分卡，便于业务人员使用
# =============================================================================
print('=' * 70)
print('⑧ 评分映射与评分卡')
print('=' * 70)

# ---------- 8.1 评分参数计算（600/50 惯例） ----------
# 为什么用 600/50 惯例：行业标准，好客户 odds=50(坏账率约2%)记 600 分，odds 翻倍 +50 分，分数直观可解释。
# Factor = 50/ln2 由「600 对应 odds=50、650 对应 odds=100」两个已知点联立解出。
p0, odds0, pdo = 600, 50, 50
factor = pdo / np.log(2)
offset = p0 - factor * np.log(odds0)
intercept = model.intercept_[0]
base_score = offset - factor * intercept
coef = dict(zip(selected, model.coef_[0]))

print('评分参数：Factor = %.4f | Offset = %.4f | 基准分(base_score) = %.1f' % (factor, offset, base_score))
print('（含义：好客户 odds=50 对应 600 分，odds 翻倍 +50 分）')
print()

# ---------- 8.2 生成评分卡明细表 ----------
scorecard_rows = []
for c in selected:
    grp = woe_detail[c]
    for bin_label, row in grp.iterrows():
        woe_val = row['WOE']
        # 为什么得分带负号：逻辑回归输出的是违约 log-odds(坏/好)，评分要用「好/坏」的 log-odds，
        # 二者差一个负号，故每档得分取负号（WOE 高=风险高=扣分）。
        score = -factor * coef[c] * woe_val
        scorecard_rows.append({
            '变量': c, '分箱': bin_label, '样本数': int(row['total']),
            'WOE': round(woe_val, 4), '得分': round(score, 1)
        })
scorecard = pd.DataFrame(scorecard_rows)

# 分箱标签美化：term 补「个月」，与数据字典一致（36/60 个月）
scorecard.loc[scorecard['变量'] == 'term', '分箱'] = \
    scorecard.loc[scorecard['变量'] == 'term', '分箱'].astype(str) + ' 个月'

# 变量按 IV 降序、箱内按得分降序
var_order = sorted(selected, key=lambda c: -iv_map[c])
scorecard['_o'] = scorecard['变量'].map({v: i for i, v in enumerate(var_order)})
scorecard = scorecard.sort_values(['_o', '得分'], ascending=[True, False]).drop(columns='_o').reset_index(drop=True)

# ---------- 8.3 格式化得分显示（正数加 + 号） ----------
# 得分显示：正数加 + 号（模板要求 +38.5 / -23.1 样式）
def fmt_score(s):
    return ('+' if s >= 0 else '') + f'{s:.1f}'
scorecard['得分'] = scorecard['得分'].apply(fmt_score)

# 基础分行插入表头（变量=基础分，其余用 — 占位）
base_row = pd.DataFrame([{'变量': '基础分', '分箱': '—', '样本数': '—', 'WOE': '—', '得分': f'{base_score:.1f}'}])
scorecard_table = pd.concat([base_row, scorecard], ignore_index=True)

print('表 1：标准评分卡（得分>0 加分、得分<0 减分）')
print(scorecard_table.to_string(index=False))
print('打分：总分 = 基础分 + 各变量命中档位得分之和，分越高风险越低。')
scorecard_table.to_csv('评分卡_表1.csv', index=False, encoding='utf-8-sig')
print()

# ---------- 8.4 打分函数与一致性校验 ----------
def score_customer(df_, bins_map_, woe_map_, coef_, factor_, base_score_):
    total = np.full(len(df_), base_score_)
    for c in selected:
        total -= factor_ * coef_[c] * apply_bins(df_[c], bins_map_[c]).map(woe_map_[c]).astype(float).fillna(0.0)
    return total

test = test.copy()
test['score'] = score_customer(test, bins_map, woe_map, coef, factor, base_score)

prob_test = model.predict_proba(X_test)[:, 1]
odds_good = (1 - prob_test) / prob_test
diff = np.abs(test['score'].values - (offset + factor * np.log(odds_good))).max()
print('评分一致性校验：评分卡求和 vs 概率反推，最大绝对差 = %.4f' % diff)
print('测试集分数分布：')
print(test['score'].describe().round(1).to_string())
print()

# ---------- 8.5 新客户打分演示 ----------
# 写清「如何给一个新客户打分」：取测试集第 1 位客户，演示 查档→取分→加总
demo = test.iloc[0]
print('如何给一个新客户打分（以测试集第 1 位客户为例）：')
print('  ① 看每个特征落在哪个档位 → ② 查评分卡取该档得分 → ③ 总分 = 基础分 + 各档得分之和。')
demo_rows = []
for c in selected:
    bl = apply_bins(pd.Series([demo[c]]), bins_map[c]).iloc[0]   # 原始档位标签，用于查 WOE
    w = woe_map[c].get(bl, 0.0)
    bl_disp = bl + ' 个月' if c == 'term' else bl                 # 展示时 term 补「个月」
    demo_rows.append({'特征': c, '命中档位': bl_disp, '得分': fmt_score(-factor * coef[c] * w)})
demo_df = pd.DataFrame(demo_rows)
print(demo_df.to_string(index=False))
print('  基础分 %.1f + 各档得分 = 该客户总分 %.1f（与 score_customer 结果一致）；再按表 2 阈值判定通过/复核/拒绝。'
      % (base_score, demo['score']))
print()


# =============================================================================
# 步骤 ⑨：风险分层、cut-off 校准与结论
# 目的：确定拒绝/复核/通过的分数阈值，输出最终放贷决策建议
# 关键原则：cut-off 阈值仅在训练集上确定，测试集仅用于评估效果
# =============================================================================
print('=' * 70)
print('⑨ 风险分层与 cut-off 校准')
print('=' * 70)

# ---------- 9.1 按分数分 10 档，验证区分度 ----------
# 为什么按 10 档看坏账率：验证分数是否单调区分风险（分数越高坏账率越低），是评分卡有效性的直接证据。
test['score_decile'] = pd.qcut(test['score'], 10, labels=False, duplicates='drop')
decile = test.groupby('score_decile', observed=True).agg(
    样本数=('score', 'count'), 最低分=('score', 'min'), 最高分=('score', 'max'), 坏账率=(TARGET, 'mean'))
decile['坏账率'] = (decile['坏账率'] * 100).round(2)
decile = decile.reset_index().rename(columns={'score_decile': '分数档'})
decile['分数档'] = decile['分数档'] + 1
print('10 档分数 → 坏账率（验证区分度，应单调递减）：')
print(decile.to_string(index=False))
print()

# ---------- 9.2 cut-off 阈值校准（仅在训练集上确定） ----------
# 为什么 cut-off 按本组人群分位校准：本数据坏账率 19.25%，教科书 600/650 对应约 2% 坏账率，
# 直接套用会把绝大多数客户拒掉；必须按本人群分数分布定阈值（这里以 30%/70% 分位切三档）。
# 为什么阈值在训练集上定：cut-off 阈值也是「用数据学出来的量」，只能由训练集决定（防泄漏），
# 测试集只用于评估（报告各档坏账率/拒单率）。
train['score'] = score_customer(train, bins_map, woe_map, coef, factor, base_score)
q_rej, q_app = train['score'].quantile([0.3, 0.7])
print('cut-off 校准：以「自动通过上限坏账率≈10%、强制拒绝下限≈25%」为业务目标，'
      '映射到本组人群 30%/70% 分位：')
print('  拒绝     < %.0f 分' % q_rej)
print('  人工复核  %.0f ~ %.0f 分' % (q_rej, q_app))
print('  通过     > %.0f 分' % q_app)

# ---------- 9.3 应用阈值到测试集，生成决策表 ----------
def decision(s):
    return '拒绝' if s < q_rej else ('人工复核' if s < q_app else '通过')

test['decision'] = test['score'].apply(decision)

# 表 2：按模板统计「分数段 / 样本占比 / 坏账率 / 建议动作」三档
def seg_stats(mask):
    share = mask.sum() / len(test) * 100
    br = test.loc[mask, TARGET].mean() * 100
    return share, br

segments = [
    ('通过',   test['score'] >= q_app, '≥ %.0f' % q_app),
    ('人工复核', (test['score'] >= q_rej) & (test['score'] < q_app), '%.0f ~ %.0f' % (q_rej, q_app)),
    ('拒绝',   test['score'] < q_rej, '< %.0f' % q_rej),
]
table2_rows = []
seg_br = {}    # 供画图用的数值坏账率（table2 里是格式化字符串，这里保留数值）
for action, mask, seg in segments:
    share, br = seg_stats(mask)
    seg_br[action] = br
    table2_rows.append({'分数段': seg, '样本占比': f'{share:.1f}%', '坏账率': f'{br:.2f}%', '建议动作': action})
table2 = pd.DataFrame(table2_rows)

rej_rate = (test['decision'] == '拒绝').mean() * 100
app_rate = (test['decision'] == '通过').mean() * 100
print()
print('表 2：分数 → 违约率映射与 cut-off')
print(table2.to_string(index=False))
print('拒单率: %.1f%% | 直接通过率: %.1f%% | 人工复核率: %.1f%%'
      % (rej_rate, app_rate, 100 - rej_rate - app_rate))
table2.to_csv('评分卡_表2_决策.csv', index=False, encoding='utf-8-sig')
print()

# ---------- 9.4 输出放贷决策建议 ----------
print('放贷决策建议：≥%.0f 分直接通过（坏账率 %.1f%%）、<%.0f 分直接拒绝（坏账率 %.1f%%）、'
      '中间人工复核；据此可自动处理约 %.0f%% 的申请（通过+拒绝），仅 %.0f%% 需人工介入，'
      '通过人群坏账率 %.1f%% 远低于整体 19.25%%，有效控制风险。'
      % (q_app, seg_br['通过'], q_rej, seg_br['拒绝'],
         app_rate + rej_rate, 100 - app_rate - rej_rate, seg_br['通过']))
print()

# ---------- 9.5 风险分层可视化 ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].bar(decile['分数档'].astype(str), decile['坏账率'], color='crimson')
axes[0].set_xlabel('分数档（1=最低分/最高风险 → 10=最高分/最低风险）')
axes[0].set_ylabel('坏账率 %')
axes[0].set_title('10 档分数 × 坏账率（区分度）')
for x, y in enumerate(decile['坏账率']):
    axes[0].text(x, y + 0.5, f'{y:.1f}', ha='center', fontsize=8)
colors = {'通过': 'seagreen', '人工复核': 'goldenrod', '拒绝': 'crimson'}
order = ['通过', '人工复核', '拒绝']
axes[1].bar(order, [seg_br[k] for k in order], color=[colors[k] for k in order])
axes[1].set_ylabel('坏账率 %')
axes[1].set_title('通过 / 人工复核 / 拒绝 三档坏账率')
for x, k in enumerate(order):
    axes[1].text(x, seg_br[k] + 0.5, f"{seg_br[k]:.2f}%", ha='center')
plt.tight_layout()
plt.savefig('fig4_risk_stratify.png', dpi=120)


# =============================================================================
# 附录 A：模型对比（RandomForest / XGBoost）
# 目的：验证可解释的 LR 模型是否足以胜任，若树模型 AUC 相当则无需牺牲解释性
# =============================================================================
print('=' * 70)
print('附录 A：模型对比（RandomForest / XGBoost）')
print('=' * 70)

# ---------- A.1 准备树模型训练数据（One-Hot 编码） ----------
def onehot_feature_matrix(df_):
    cats = [c for c in cat_cols if c in selected]
    X = df_[selected].copy()
    for c in cats:
        X[c] = X[c].astype(str)
    return pd.get_dummies(X, columns=cats)

X_tr_raw = onehot_feature_matrix(train)
X_te_raw = onehot_feature_matrix(test)
X_tr_raw, X_te_raw = X_tr_raw.align(X_te_raw, join='left', axis=1)
X_te_raw = X_te_raw.fillna(0)

# ---------- A.2 随机森林 ----------
# 为什么做模型对比：检验树模型是否显著优于 LR；若 AUC 相当，则证明「可解释的 LR」足以胜任，
# 不必牺牲解释性（评分卡必须能逐条向客户解释得分构成）。
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=50, n_jobs=-1, random_state=SEED)
rf.fit(X_tr_raw, y_train)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_te_raw)[:, 1])

# ---------- A.3 XGBoost / LightGBM / HistGradientBoosting 备选 ----------
# 优先用题目要求的 XGBoost；不可用则退到 LightGBM，再退到 sklearn HistGradientBoosting
try:
    import xgboost as xgb
    gbm = xgb.XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5,
                            min_child_weight=50, subsample=0.8, colsample_bytree=0.8,
                            eval_metric='auc', random_state=SEED)
    gbm.fit(X_tr_raw, y_train)
    auc_gbm = roc_auc_score(y_test, gbm.predict_proba(X_te_raw)[:, 1])
    gbm_name = 'XGBoost'
except Exception:
    try:
        import lightgbm as lgb
        gbm = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                                 min_child_samples=50, random_state=SEED, verbose=-1)
        gbm.fit(X_tr_raw, y_train)
        auc_gbm = roc_auc_score(y_test, gbm.predict_proba(X_te_raw)[:, 1])
        gbm_name = 'LightGBM'
    except Exception:
        from sklearn.ensemble import HistGradientBoostingClassifier
        gbm = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                             max_leaf_nodes=31, min_samples_leaf=50, random_state=SEED)
        gbm.fit(X_tr_raw, y_train)
        auc_gbm = roc_auc_score(y_test, gbm.predict_proba(X_te_raw)[:, 1])
        gbm_name = 'HistGradientBoosting'

# ---------- A.4 输出对比结果 ----------
print('模型对比（同一清洗/特征，70/30 划分）：')
print('  Logistic Regression(WOE)  AUC = %.4f  ← 主模型' % auc)
print('  RandomForest              AUC = %.4f' % auc_rf)
print('  %-26s AUC = %.4f' % (gbm_name, auc_gbm))
print('说明：树模型 AUC 与 LR 基本相当（略高或持平），但评分卡要求可解释、能逐条给客户'
      '解释得分构成，监管也偏好单调可解释模型，故仍以 LR 为主模型；树模型可作为辅助校验。')
print()


# =============================================================================
# 附录 B：时间切分验证（按 issue_month 前 80% 训练 / 后 20% 测试）
# 目的：模拟「用历史预测未来」，检验模型在样本外时间上的稳定性
# 关键原则：所有拟合参数（清洗、特征、分箱、WOE、LR）均在时间训练集上重新计算
# =============================================================================
print('=' * 70)
print('附录 B：时间切分验证（按 issue_month）')
print('=' * 70)

# ---------- B.1 按月份切分数据集 ----------
# 为什么做时间切分：随机切分可能让「未来信息」泄漏；按放款月份前 80% 训练/后 20% 测试，
# 完全模拟「用历史预测未来」，检验模型在样本外时间的稳定性。
df_raw2 = df_raw.copy()
df_raw2['ym'] = df_raw2['issue_month'].str.replace('-', '').astype(int)
df_raw2 = df_raw2.sort_values('ym').reset_index(drop=True)
n_ts = len(df_raw2)
ts_cut = int(n_ts * 0.8)
tr_raw_ts, te_raw_ts = df_raw2.iloc[:ts_cut], df_raw2.iloc[ts_cut:]
print('时间切分：训练 %d 行(%.2f) | 测试 %d 行(%.2f)'
      % (len(tr_raw_ts), len(tr_raw_ts) / n_ts, len(te_raw_ts), len(te_raw_ts) / n_ts))
print('  训练期 issue_month:', tr_raw_ts['issue_month'].min(), '~', tr_raw_ts['issue_month'].max())
print('  测试期 issue_month:', te_raw_ts['issue_month'].min(), '~', te_raw_ts['issue_month'].max())

# ---------- B.2 在时间训练集上重新执行全部拟合流程 ----------
# 同样 fit/apply：清洗、特征、分箱、WOE、LR 全部只在时间训练集拟合
cp_ts = clean_fit(tr_raw_ts)
tr_c_ts = clean_apply(tr_raw_ts, cp_ts)
te_c_ts = clean_apply(te_raw_ts, cp_ts)
fp_ts = feat_fit(tr_c_ts)
tr_ts = feat_apply(tr_c_ts, fp_ts)
te_ts = feat_apply(te_c_ts, fp_ts)

bins_ts = {c: build_bins(tr_ts[c], kind_map[c]) for c in feature_cols}
woe_ts = {}
iv_ts = {}
for c in feature_cols:
    grp, iv = calc_woe_iv(tr_ts.assign(**{'__bin__': apply_bins(tr_ts[c], bins_ts[c])}), '__bin__')
    woe_ts[c] = grp['WOE'].to_dict()
    iv_ts[c] = iv

# ---------- B.3 特征选择必须在时间训练集上重新做 ----------
# 关键：特征选择（IV 筛选 + 共线性剔除）必须在时间训练集上重新做，不能继承随机划分的 selected。
# 否则随机训练集里已混入后 20% 的样本，等于特征选择「看过未来」，造成特征选择信息污染。
def woe_encode_ts(df_, c):
    return apply_bins(df_[c], bins_ts[c]).map(woe_ts[c]).astype(float).fillna(0.0)

sel_iv_ts = [c for c in feature_cols if iv_ts[c] >= 0.02]
Xtr_ts_all = pd.DataFrame({c: woe_encode_ts(tr_ts, c) for c in sel_iv_ts})
corr_ts = Xtr_ts_all.corr()
drop_coll_ts = set()
for i in range(len(sel_iv_ts)):
    for j in range(i + 1, len(sel_iv_ts)):
        a, b = sel_iv_ts[i], sel_iv_ts[j]
        if abs(corr_ts.loc[a, b]) > 0.9 and a not in drop_coll_ts and b not in drop_coll_ts:
            drop_coll_ts.add(a if iv_ts[a] < iv_ts[b] else b)
sel_ts = [c for c in sel_iv_ts if c not in drop_coll_ts]
print('时间训练集重新 IV 筛选 + 共线性剔除 → 入模特征 %d 个（随机划分入模 %d 个）'
      % (len(sel_ts), len(selected)))
print('  特征集合是否一致：%s' % ('完全一致' if set(sel_ts) == set(selected) else '有差异 → ' + str(sorted(set(sel_ts) ^ set(selected)))))

# ---------- B.4 时间切分模型训练与评估 ----------
Xtr_ts = pd.DataFrame({c: woe_encode_ts(tr_ts, c) for c in sel_ts})
Xte_ts = pd.DataFrame({c: woe_encode_ts(te_ts, c) for c in sel_ts})
m_ts = LogisticRegression(max_iter=2000).fit(Xtr_ts, tr_ts[TARGET].values)
p_ts = m_ts.predict_proba(Xte_ts)[:, 1]
auc_ts = roc_auc_score(te_ts[TARGET].values, p_ts)
fpr_ts, tpr_ts, _ = roc_curve(te_ts[TARGET].values, p_ts)
ks_ts = (tpr_ts - fpr_ts).max()
print('时间切分（OOT 样本外）测试 AUC = %.4f | KS = %.4f' % (auc_ts, ks_ts))
print('对比随机划分 AUC = %.4f | KS = %.4f；二者接近，说明模型时间上稳定（无过拟合/泄漏）。' % (auc, ks))
print()


# =============================================================================
# 附录 C：衍生特征对模型表现的提升
# 目的：量化新特征的边际贡献，证明特征工程确实提升了模型表现
# =============================================================================
print('=' * 70)
print('附录 C：衍生特征对模型表现的提升')
print('=' * 70)

# ---------- C.1 对比有无衍生特征时的 AUC ----------
# 为什么对比有无衍生特征：量化新特征的边际贡献，证明特征工程确实提升了模型表现。
base_features = [c for c in selected if c not in DERIVED]
X_tr_base = X_train[base_features]
X_te_base = X_test[base_features]
m_base = LogisticRegression(max_iter=2000).fit(X_tr_base, y_train)
auc_base = roc_auc_score(y_test, m_base.predict_proba(X_te_base)[:, 1])

# ---------- C.2 输出提升结果 ----------
print('不含衍生特征 LR AUC = %.4f' % auc_base)
print('含衍生特征 LR AUC   = %.4f（提升 %.4f）' % (auc, auc - auc_base))
print('说明：loan_to_income 的 IV=%.4f 排名第 3，本身有强区分力；但因与 dti/annual_inc/loan_amnt'
      ' 信息重叠，入模后整体 AUC 提升有'
      '限。衍生特征的价值更多在于业务可解释性'
      '（相对负担/杠杆比绝对金额更可比）。' % iv_map['loan_to_income'])


# =============================================================================
# PPT 图表生成
# =============================================================================
print('=' * 70)
print('PPT 图表生成')
print('=' * 70)

from matplotlib.colors import LinearSegmentedColormap, Normalize

# ---------- 配色方案 ----------
# 顺序色带（蓝 浅→深，编码「量级/大小」）；发散色带（蓝→灰→红，编码「正负/极性」）
cmap_blue = LinearSegmentedColormap.from_list('blue_ramp', ['#cde2fb', '#86b6ef', '#3987e5', '#184f95', '#0d366b'])
cmap_div = LinearSegmentedColormap.from_list('div', ['#184f95', '#86b6ef', '#f0efec', '#ec835a', '#1baf7a', '#e34948'])
INK = '#0b0b0b'
MUTED = '#898781'

# ---------- 图表 1：目标变量分布（环形图） ----------
good = int((df_raw[TARGET] == 0).sum())
bad = int((df_raw[TARGET] == 1).sum())
fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    [good, bad], labels=['好客户（履约）', '坏客户（违约）'], colors=['#1baf7a', '#e34948'],
    autopct='%.2f%%', startangle=90, pctdistance=0.72,
    wedgeprops=dict(width=0.42, edgecolor='white', linewidth=2),
    textprops=dict(color=INK, fontsize=11))
for at in autotexts:
    at.set_color('white'); at.set_fontweight('bold'); at.set_fontsize(11)
ax.text(0, 0, f'{good + bad:,}\n总样本', ha='center', va='center', fontsize=12, color=MUTED)
ax.set_title('目标变量分布（坏账率 19.25%）', fontsize=13)
plt.tight_layout()
plt.savefig('fig5_target_dist.png', dpi=130)

# ---------- 图表 2：grade × 违约率（单调上升，顺序渐变） ----------
grade_br = df_raw.groupby('grade')[TARGET].mean().sort_index() * 100
norm_g = Normalize(grade_br.min(), grade_br.max())
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(grade_br.index, grade_br.values,
              color=[cmap_blue(norm_g(v)) for v in grade_br.values], width=0.62)
for b, v in zip(bars, grade_br.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f'{v:.1f}%', ha='center', fontsize=9, color=INK)
ax.set_xlabel('grade（平台风险等级，A 最优 → G 最差）')
ax.set_ylabel('坏账率 %')
ax.set_title('grade × 违约率（单调递增 → 强区分力）', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig6_grade_badrate.png', dpi=130)

# ---------- 图表 3：IV 排序（保留 IV≥0.01 的特征） ----------
iv_top = iv_df[iv_df['IV'] >= 0.02].iloc[::-1].reset_index(drop=True)
norm_iv = Normalize(iv_top['IV'].min(), iv_top['IV'].max())
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.barh(iv_top['特征'], iv_top['IV'], color=[cmap_blue(norm_iv(v)) for v in iv_top['IV']])
for b, v in zip(bars, iv_top['IV']):
    ax.text(v + 0.006, b.get_y() + b.get_height() / 2, f'{v:.3f}', va='center', fontsize=8, color=INK)
ax.axvline(0.02, color='#e34948', ls='--', lw=1, label='IV 阈值 0.02')
ax.set_xlabel('IV（信息价值）')
ax.set_title('IV 排序（保留 IV≥0.01 的特征）', fontsize=13)
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig7_iv_rank.png', dpi=130)

# ---------- 图表 4：grade 各档 WOE（发散色带） ----------
grp_grade = woe_detail['sub_grade']
woe_g = grp_grade['WOE']
max_abs = max(abs(woe_g.min()), abs(woe_g.max()))
norm_w = Normalize(-max_abs, max_abs)
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(woe_g.index, woe_g.values, color=[cmap_div(norm_w(v)) for v in woe_g.values], width=0.6)
ax.axhline(0, color='#c3c2b7', lw=1)
for b, v in zip(bars, woe_g.values):
    ax.text(b.get_x() + b.get_width() / 2, v + (0.03 if v >= 0 else -0.12), f'{v:.2f}',
            ha='center', fontsize=8, color=INK)
ax.set_xlabel('sub_grade')
ax.set_ylabel('WOE = ln(坏占比/好占比)')
ax.set_title('sub_grade 各档 WOE（WOE>0 高风险、WOE<0 低风险）', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig8_woe_grade.png', dpi=130)

# ---------- 图表 5：评分分布 + cut-off 三档 ----------
fig, ax = plt.subplots(figsize=(7.5, 4.5))
counts, bins, patches = ax.hist(test['score'], bins=45, alpha=0.95, edgecolor='white', linewidth=0.4)
for i, p in enumerate(patches):
    p.set_facecolor(cmap_blue(1.0 - 0.85 * i / max(len(patches) - 1, 1)))
ax.axvline(q_rej, color='#e34948', ls='--', lw=1.4, label=f'拒绝 < {q_rej:.0f}')
ax.axvline(q_app, color='#1baf7a', ls='--', lw=1.4, label=f'通过 > {q_app:.0f}')
ax.set_xlabel('信用分')
ax.set_ylabel('客户数')
ax.set_title('测试集评分分布（颜色越深风险越高）', fontsize=13)
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig9_score_dist.png', dpi=130)

# ---------- 图表 6：入模特征 WOE 相关性热力图 ----------
corr_mat = X_train[selected].corr()
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr_mat, cmap=cmap_div, center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', linewidths=0.5, linecolor='white',
            square=True, cbar_kws={'shrink': 0.8}, ax=ax, annot_kws={'size': 7})
ax.set_title('入模特征 WOE 相关性（蓝=负相关、红=正相关）', fontsize=13)
plt.tight_layout()
plt.savefig('fig10_corr_heatmap.png', dpi=130)

print()
print('=' * 70)
print('全部完成。输出文件：fig1~fig10.png、fig_categorical.png、评分卡_表1.csv、评分卡_表2_决策.csv')
print('=' * 70)

# =============================================================================
# 补充图表：策略 4（sub_grade）与策略 8（IV 阈值敏感性、WOE 相关性）的可视化
# 说明：以下代码只新增绘图，不改变前述任何模型、筛选、评分卡与评估逻辑。
# 所有图均直接使用当前版本已经计算得到的结果。
# =============================================================================
print('=' * 70)
print('补充图表：sub_grade、IV 阈值敏感性、模型系数与时间稳定性')
print('=' * 70)

# ---------- 补充图表 1：sub_grade × 坏账率 ----------
# sub_grade 是当前主模型使用的细粒度评级变量，比 grade 包含更多风险层级信息。
sub_grade_br = (df_raw.groupby('sub_grade')[TARGET]
                .agg(['mean', 'count'])
                .sort_index())
sub_grade_br['坏账率'] = sub_grade_br['mean'] * 100

fig, ax = plt.subplots(figsize=(12, 5.2))
bar_colors = [cmap_blue((v - sub_grade_br['坏账率'].min()) /
                        max(sub_grade_br['坏账率'].max() - sub_grade_br['坏账率'].min(), 1e-9))
              for v in sub_grade_br['坏账率']]
bars = ax.bar(sub_grade_br.index, sub_grade_br['坏账率'],
              color=bar_colors, width=0.72)
for b, v in zip(bars, sub_grade_br['坏账率']):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.25,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=7)
ax.set_xlabel('sub_grade（细粒度风险等级）')
ax.set_ylabel('坏账率 %')
ax.set_title('sub_grade × 坏账率（细粒度评级风险差异）', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('fig11_sub_grade_badrate.png', dpi=150, bbox_inches='tight')
plt.close()

# ---------- 补充图表 2：IV 阈值敏感性分析 ----------
# 横轴为候选 IV 阈值，纵轴为训练集 5 折 CV AUC。
# 该图用于说明最终阈值并非由测试集效果反向选择，而是由训练集交叉验证确定。
fig, ax = plt.subplots(figsize=(7.5, 4.8))
plot_threshold = threshold_df.sort_values('IV阈值')
line = ax.plot(plot_threshold['IV阈值'], plot_threshold['训练集5折CV_AUC'],
               marker='o', lw=2.0, markersize=6, label='5 折 CV AUC')
ax.axvline(0.02, color='#e34948', ls='--', lw=1.3, label='IV = 0.02')
ax.axvline(best_threshold, color='#1baf7a', ls='-.', lw=1.3,
           label=f'最优阈值 = {best_threshold:.3f}')
for x, y in zip(plot_threshold['IV阈值'], plot_threshold['训练集5折CV_AUC']):
    ax.text(x, y + 0.0008, f'{y:.4f}', ha='center', va='bottom', fontsize=8)
ax.set_xlabel('IV 筛选阈值')
ax.set_ylabel('训练集 5 折 CV AUC')
ax.set_title('IV 阈值敏感性分析（训练集交叉验证）', fontsize=13)
ax.legend(frameon=True)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('fig12_iv_threshold_cv.png', dpi=150, bbox_inches='tight')
plt.close()

# ---------- 补充图表 3：最终模型 WOE 逻辑回归系数 ----------
# 系数绝对值越大，表示该变量对违约 log-odds 的边际影响越明显。
coef_plot = (pd.DataFrame({'特征': selected, '系数': model.coef_[0]})
             .sort_values('系数'))
fig, ax = plt.subplots(figsize=(8, max(5.0, 0.34 * len(coef_plot) + 1.5)))
coef_colors = [('#e34948' if v > 0 else '#184f95') for v in coef_plot['系数']]
bars = ax.barh(coef_plot['特征'], coef_plot['系数'], color=coef_colors, height=0.62)
ax.axvline(0, color='#777777', lw=1)
for b, v in zip(bars, coef_plot['系数']):
    offset_x = 0.01 if v >= 0 else -0.01
    ha = 'left' if v >= 0 else 'right'
    ax.text(v + offset_x, b.get_y() + b.get_height() / 2,
            f'{v:.3f}', va='center', ha=ha, fontsize=8)
ax.set_xlabel('逻辑回归系数')
ax.set_title('最终评分卡模型逻辑回归系数', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.savefig('fig13_model_coefficients.png', dpi=150, bbox_inches='tight')
plt.close()

# ---------- 补充图表 4：随机划分与时间切分 AUC 对比 ----------
# 用于直观展示随机测试集与时间外推测试集的模型区分能力，辅助判断时间稳定性。
auc_compare = pd.DataFrame({
    '验证方式': ['随机划分测试集', '时间外推测试集'],
    'AUC': [auc, auc_ts]
})
fig, ax = plt.subplots(figsize=(7.5, 4.8))
bars = ax.bar(auc_compare['验证方式'], auc_compare['AUC'],
              color=['#3987e5', '#184f95'], width=0.58)
ax.set_ylim(max(0.5, auc_compare['AUC'].min() - 0.04),
            min(1.0, auc_compare['AUC'].max() + 0.04))
for b, v in zip(bars, auc_compare['AUC']):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.005,
            f'{v:.4f}', ha='center', fontsize=10)
ax.set_ylabel('AUC')
ax.set_title('随机划分与时间外推验证的 AUC 对比', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('fig14_auc_stability.png', dpi=150, bbox_inches='tight')
plt.close()

# ---------- 补充图表 5：有无衍生特征的 AUC 对比 ----------
feature_auc = pd.DataFrame({
    '模型': ['不含衍生特征', '含衍生特征'],
    'AUC': [auc_base, auc]
})
fig, ax = plt.subplots(figsize=(7.5, 4.8))
bars = ax.bar(feature_auc['模型'], feature_auc['AUC'],
              color=['#86b6ef', '#184f95'], width=0.58)
ax.set_ylim(max(0.5, feature_auc['AUC'].min() - 0.04),
            min(1.0, feature_auc['AUC'].max() + 0.04))
for b, v in zip(bars, feature_auc['AUC']):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.005,
            f'{v:.4f}', ha='center', fontsize=10)
ax.set_ylabel('测试集 AUC')
ax.set_title('衍生特征对模型区分能力的影响', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('fig15_derived_feature_auc.png', dpi=150, bbox_inches='tight')
plt.close()

print('补充图表已生成：')
for f in [
    'fig11_sub_grade_badrate.png',
    'fig12_iv_threshold_cv.png',
    'fig13_model_coefficients.png',
    'fig14_auc_stability.png',
    'fig15_derived_feature_auc.png'
]:
    print('  ', f)
print('=' * 70)

# =============================================================================
# 高阶渐变可视化（用于答辩 PPT，风格统一、渐变复杂）
# 说明：以下代码只新增绘图，不改变任何模型、筛选、评分卡与评估逻辑。
# =============================================================================
print('=' * 70)
print('高阶渐变可视化（答辩 PPT 用）')
print('=' * 70)

from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.collections import PolyCollection
from scipy.stats import gaussian_kde

# ---------- 追加高级渐变色带 ----------
cmap_aurora = LinearSegmentedColormap.from_list('aurora',
    ['#0d1b3e', '#1b3a6b', '#2e6f95', '#5aa9c9', '#8ed1b5', '#eef2a0'])          # 极光蓝绿
cmap_sunset = LinearSegmentedColormap.from_list('sunset',
    ['#2b0a3d', '#6b1f5e', '#c0446f', '#e9785a', '#f4a261', '#f7d488'])          # 落日紫橙
cmap_red_ramp = LinearSegmentedColormap.from_list('red_ramp',
    ['#fde3e3', '#f6a8a8', '#e34948', '#b3202a', '#7a1420'])                     # 红系渐变（风险）
cmap_grn_ramp = LinearSegmentedColormap.from_list('grn_ramp',
    ['#d9f2e5', '#8fd6b0', '#1baf7a', '#0f7a52', '#0a4a33'])                     # 绿系渐变（安全）
cmap_div_smooth = LinearSegmentedColormap.from_list('div_smooth',
    ['#14346b', '#2f5fb0', '#7fb2e5', '#e8eef6', '#f2d9c8', '#ec835a', '#c0392b'])  # 平滑发散

FEAT_CN = {
    'sub_grade': '细粒度评级', 'int_rate': '利率', 'grade': '评级',
    'term': '期限', 'loan_amnt': '贷款金额', 'installment': '月供',
    'annual_inc': '年收入', 'dti': '负债收入比', 'emp_length': '工作年限',
    'revol_bal': '循环余额', 'revol_util': '循环使用率', 'open_acc': '账户数',
    'total_acc': '总账户数', 'mths_since_earliest_cr_line': '信用历史',
    'mths_since_last_delinq': '距上次逾期', 'loan_to_income': '贷款收入比',
    'repay_burden': '还款负担率', 'revol_bal_to_income': '循环负债收入比',
    'tot_bal_to_income': '总余额收入比', 'credit_line_util': '额度使用率',
    'installment_to_loan': '年还款/本金', 'home_ownership': '住房情况',
    'verification_status': '核验状态', 'purpose': '贷款用途',
    'addr_state': '所在州', 'initial_list_status': '初始列表状态',
    'delinq_2yrs': '近2年逾期', 'inq_last_6mths': '近6月查询',
    'pub_rec': '公共记录', 'acc_now_delinq': '当前逾期账户',
    'collections_12_mths_ex_med': '近12月催收', 'tot_coll_amt': '催收总额',
    'high_util_flag': '高使用率标记', 'delinq_flag': '逾期标记',
    'inq_flag': '查询标记', 'pub_rec_flag': '公共记录标记',
}


def _cn(c):
    return FEAT_CN.get(c, c)


def _save(fig, name, dpi=150):
    fig.savefig(name, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)


def _fill_under_gradient(ax, x, y, cmap, vmin, vmax, baseline=0.0, alpha=0.85, n=70):
    """在曲线 y 下方用纵向渐变填充（自 baseline 到曲线）。"""
    norm = Normalize(vmin, vmax)
    xs = np.linspace(np.nanmin(x), np.nanmax(x), 1200)
    ys = np.interp(xs, x, y)
    levels = np.linspace(baseline, np.nanmax(ys), n)
    for k in range(n - 1):
        lo, hi = levels[k], levels[k + 1]
        mask = ys >= lo
        if mask.sum() < 2:
            continue
        ax.fill_between(xs, lo, np.where(mask, ys, lo), where=mask,
                        color=cmap(norm((lo + hi) / 2)), lw=0, alpha=alpha)


def _vgradient_bar(ax, x, width, ymin, ymax, cmap, vmin=None, vmax=None, n=100):
    """在 (x±width/2, ymin~ymax) 区域绘制自下而上的垂直渐变矩形。"""
    vmin = ymin if vmin is None else vmin
    vmax = ymax if vmax is None else vmax
    norm = Normalize(vmin, vmax)
    ys = np.linspace(ymin, ymax, n + 1)
    verts, cols = [], []
    for k in range(n):
        y0, y1 = ys[k], ys[k + 1]
        verts.append([(x - width / 2, y0), (x + width / 2, y0),
                      (x + width / 2, y1), (x - width / 2, y1)])
        cols.append(cmap(norm((y0 + y1) / 2)))
    ax.add_collection(PolyCollection(verts, facecolors=cols, edgecolors='none'))


# ---------- 高阶图 1：ROC 曲线（渐变填充，随机划分 + 时间外推叠加） ----------
fig, ax = plt.subplots(figsize=(6.6, 6.2))
_fill_under_gradient(ax, fpr, tpr, cmap_aurora, 0.0, 1.0, baseline=0.0, alpha=0.9, n=70)
ax.plot(fpr, tpr, color='#0d1b3e', lw=2.4, label=f'随机划分  AUC = {auc:.4f}')
ax.plot(fpr_ts, tpr_ts, color='#e34948', lw=2.0, ls='--', label=f'时间外推  AUC = {auc_ts:.4f}')
ax.plot([0, 1], [0, 1], color=MUTED, lw=1.2, ls=':')
ax.set_xlabel('假正率 FPR'); ax.set_ylabel('真正率 TPR')
ax.set_title('ROC 曲线（渐变填充 + 时间外推对比）', fontsize=13)
ax.legend(loc='lower right', frameon=False)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)
plt.tight_layout()
_save(fig, 'fig16_roc_gradient.png')

# ---------- 高阶图 2：KS 曲线（TPR/FPR 之间渐变区域 = 模型分离度） ----------
fig, ax = plt.subplots(figsize=(6.6, 5.2))
x_ks = np.arange(len(tpr))
ax.fill_between(x_ks, tpr, fpr, color='#3987e5', alpha=0.15)
ax.plot(x_ks, tpr, color='#e34948', lw=2.2, label='TPR（累计坏账率）')
ax.plot(x_ks, fpr, color='#184f95', lw=2.2, label='FPR（累计好账误判率）')
ks_idx = np.argmax(tpr - fpr)
ax.axvline(ks_idx, color=MUTED, ls='--', lw=1.2)
ax.annotate(f'KS = {ks:.4f}', xy=(ks_idx, (tpr[ks_idx] + fpr[ks_idx]) / 2),
            xytext=(ks_idx + len(tpr) * 0.05, (tpr[ks_idx] + fpr[ks_idx]) / 2 + 0.08),
            fontsize=11, color=INK, arrowprops=dict(arrowstyle='->', color=INK))
ax.set_xlabel('样本（按风险从高到低排序）'); ax.set_ylabel('累计比例')
ax.set_title('KS 曲线（TPR 与 FPR 之间为模型分离度）', fontsize=13)
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)
plt.tight_layout()
_save(fig, 'fig17_ks_gradient.png')

# ---------- 高阶图 3：好/坏客户评分分布（KDE 渐变填充 + cut-off） ----------
s_bad = test.loc[test[TARGET] == 1, 'score'].values
s_good = test.loc[test[TARGET] == 0, 'score'].values
xs_kde = np.linspace(test['score'].min(), test['score'].max(), 600)
try:
    kde_bad = gaussian_kde(s_bad)(xs_kde)
    kde_good = gaussian_kde(s_good)(xs_kde)
except Exception:
    kde_bad, _ = np.histogram(s_bad, bins=60, density=True)
    kde_good, _ = np.histogram(s_good, bins=60, density=True)
    xs_kde = np.linspace(test['score'].min(), test['score'].max(), 60)
fig, ax = plt.subplots(figsize=(7.6, 5.0))
_kmax = max(kde_bad.max(), kde_good.max())
_fill_under_gradient(ax, xs_kde, kde_bad, cmap_red_ramp, 0, _kmax, baseline=0, alpha=0.55, n=60)
_fill_under_gradient(ax, xs_kde, kde_good, cmap_grn_ramp, 0, _kmax, baseline=0, alpha=0.55, n=60)
ax.plot(xs_kde, kde_good, color='#0f7a52', lw=2.2, label='好客户（履约）')
ax.plot(xs_kde, kde_bad, color='#b3202a', lw=2.2, label='坏客户（违约）')
ax.axvline(q_rej, color='#e34948', ls='--', lw=1.4, label=f'拒绝 < {q_rej:.0f}')
ax.axvline(q_app, color='#1baf7a', ls='--', lw=1.4, label=f'通过 > {q_app:.0f}')
ax.set_xlabel('信用分'); ax.set_ylabel('密度')
ax.set_title('好/坏客户评分分布（KDE 渐变，分数越高风险越低）', fontsize=13)
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)
plt.tight_layout()
_save(fig, 'fig18_score_goodbad_kde.png')

# ---------- 高阶图 4：评分卡热力图（发散渐变，行=变量，列=分箱档位） ----------
sc_num = pd.DataFrame(scorecard_rows)   # 数值型评分卡（得分未格式化）
top12 = sorted(selected, key=lambda c: -iv_map[c])[:12]
sc_top = sc_num[sc_num['变量'].isin(top12)].copy()
sc_top['bin_rank'] = sc_top.groupby('变量').cumcount()
pivot = sc_top.pivot(index='变量', columns='bin_rank', values='得分')
pivot = pivot.reindex([c for c in top12 if c in pivot.index])
ann = pivot.round(1).astype(object)
ann = ann.where(pivot.notna(), '')
fig, ax = plt.subplots(figsize=(max(10, 0.55 * pivot.shape[1] + 3), max(5, 0.42 * pivot.shape[0] + 2)))
sns.heatmap(pivot, cmap=cmap_div_smooth, center=0,
            annot=ann.values, fmt='', linewidths=1.2, linecolor='white',
            square=False, cbar_kws={'label': '得分', 'shrink': 0.8}, ax=ax,
            annot_kws={'size': 7})
ax.set_xticklabels([f'档 {int(c) + 1}' for c in pivot.columns], rotation=0)
ax.set_yticklabels([_cn(c) for c in pivot.index], rotation=0)
ax.set_xlabel('分箱档位（从左到右风险升高）')
ax.set_title('评分卡热力图（加分=低风险、减分=高风险）', fontsize=13)
plt.tight_layout()
_save(fig, 'fig19_scorecard_heatmap.png')

# ---------- 高阶图 5：关键特征 WOE 单调性小多图 ----------
lattice_feats = sorted(selected, key=lambda c: -iv_map[c])[:9]
ncols = 3
nrows = int(np.ceil(len(lattice_feats) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.1 * ncols, 3.0 * nrows))
axes = np.array(axes).ravel()
for ax, c in zip(axes, lattice_feats):
    grp = woe_detail[c]
    woe = grp['WOE']
    mabs = max(abs(woe.min()), abs(woe.max()))
    norm_w = Normalize(-mabs, mabs)
    colors = [cmap_div_smooth(norm_w(v)) for v in woe]
    xs = np.arange(len(woe))
    ax.bar(xs, woe.values, color=colors, width=0.7)
    ax.axhline(0, color='#c3c2b7', lw=1)
    ax.plot(xs, woe.values, color=INK, lw=1.2, marker='o', markersize=3)
    ax.set_xticks(xs)
    ax.set_xticklabels(list(grp.index), rotation=45, fontsize=6)
    ax.set_title(f'{_cn(c)}  (IV={iv_map[c]:.3f})', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.2)
for ax in axes[len(lattice_feats):]:
    ax.axis('off')
fig.suptitle('关键特征 WOE 单调性（WOE 越高风险越高）', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.95])
_save(fig, 'fig20_woe_lattice.png')

# ---------- 高阶图 6：模型综合指标雷达图 ----------
auto_rate = (app_rate + rej_rate) / 100.0
pass_safe = 1.0 - seg_br['通过'] / (df_raw[TARGET].mean() * 100)
time_stab = 1.0 - abs(auc - auc_ts)
radar_labels = ['区分度\nAUC', '分离度\nKS', '自动决策率', '通过人群\n安全性', '时间\n稳定性', '可解释性']
radar_vals = np.clip([auc, ks, auto_rate, pass_safe, time_stab, 0.95], 0, 1)
n_rad = len(radar_labels)
angles = np.linspace(0, 2 * np.pi, n_rad, endpoint=False).tolist()
angles += angles[:1]
vals_c = np.concatenate([radar_vals, radar_vals[:1]])
fig, ax = plt.subplots(figsize=(6.2, 6.2), subplot_kw=dict(polar=True))
for r in np.arange(0.2, 1.01, 0.2):
    ax.plot(angles, [r] * len(angles), color='#d9d8d2', lw=0.8)
for a in angles[:-1]:
    ax.plot([a, a], [0, 1], color='#d9d8d2', lw=0.8)
_norm = Normalize(0, 1)
for k in range(60):
    r0, r1 = k / 60, (k + 1) / 60
    ax.fill_between(angles, r0 * np.ones_like(angles), r1 * np.ones_like(angles),
                    color=cmap_aurora(_norm((r0 + r1) / 2)), alpha=0.5)
ax.fill(angles, vals_c, color=cmap_aurora(0.75), alpha=0.4)
ax.plot(angles, vals_c, color=INK, lw=2.2, marker='o', markersize=4)
for a, v in zip(angles[:-1], radar_vals):
    ax.text(a, v + 0.07, f'{v:.2f}', ha='center', fontsize=8, color=INK)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=8)
ax.set_ylim(0, 1); ax.set_yticks([])
ax.set_title('模型综合能力雷达图', fontsize=13, pad=22)
plt.tight_layout()
_save(fig, 'fig21_radar_metrics.png')

# ---------- 高阶图 7：累计增益曲线（渐变填充） ----------
order = np.argsort(-prob)
cum_bad = np.cumsum(y_test[order])
x_gain = np.arange(1, len(order) + 1) / len(order)
gain = cum_bad / cum_bad[-1]
fig, ax = plt.subplots(figsize=(6.8, 5.2))
_fill_under_gradient(ax, x_gain, gain, cmap_sunset, 0, 1, baseline=0, alpha=0.9, n=70)
ax.plot(x_gain, gain, color='#6b1f5e', lw=2.2, label='模型（按风险排序）')
ax.plot([0, 1], [0, 1], color=MUTED, lw=1.3, ls=':', label='随机猜测')
for q in [0.1, 0.2, 0.3]:
    gv = np.interp(q, x_gain, gain)
    ax.axvline(q, color='#c3c2b7', lw=0.8, ls='--')
    ax.axhline(gv, color='#c3c2b7', lw=0.8, ls='--')
    ax.plot(q, gv, 'o', color='#c0446f', ms=5)
    ax.annotate(f'前{int(q * 100)}%样本\n捕获{int(gv * 100)}%违约', xy=(q, gv),
                xytext=(q + 0.03, gv - 0.12), fontsize=8, color=INK)
ax.set_xlabel('样本占比（按预测风险从高到低）'); ax.set_ylabel('累计捕获违约占比')
ax.set_title('累计增益曲线（Lift / Gains）', fontsize=13)
ax.legend(frameon=False, loc='upper left')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)
plt.tight_layout()
_save(fig, 'fig22_lift_curve.png')

# ---------- 高阶图 8：模型对比（渐变柱状 + 数值标签） ----------
model_names = ['Logistic\nRegression', 'RandomForest', gbm_name.replace('HistGradientBoosting', 'HistGBM')]
model_aucs = [auc, auc_rf, auc_gbm]
fig, ax = plt.subplots(figsize=(7, 4.8))
for i, (nm, va) in enumerate(zip(model_names, model_aucs)):
    _vgradient_bar(ax, i, 0.5, 0, va, cmap_aurora,
                   vmin=min(model_aucs) * 0.9, vmax=max(model_aucs) * 1.01, n=80)
    ax.text(i, va + 0.004, f'{va:.4f}', ha='center', fontsize=11, color=INK)
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('测试集 AUC')
ax.set_title('模型区分能力对比（LR 可解释，足以胜任）', fontsize=13)
ax.set_ylim(min(model_aucs) * 0.85, max(model_aucs) * 1.04)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
_save(fig, 'fig23_model_compare.png')

# ---------- 高阶图 9：缺失值分析（渐变柱状，仅显示缺失>0 的列） ----------
miss2 = df_raw.isna().sum()
miss2 = miss2[miss2 > 0].sort_values(ascending=True)
miss2_pct = (miss2 / len(df_raw) * 100)
fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(miss2) + 2))
norm_miss = Normalize(miss2_pct.min(), miss2_pct.max())
colors = [cmap_red_ramp(norm_miss(v)) for v in miss2_pct]
bars = ax.barh([_cn(c) for c in miss2.index], miss2_pct.values, color=colors, height=0.6)
for b, v in zip(bars, miss2_pct.values):
    ax.text(v + 0.3, b.get_y() + b.get_height() / 2, f'{v:.2f}%', va='center', fontsize=8, color=INK)
ax.set_xlabel('缺失率 %')
ax.set_title('缺失值分析（仅列出缺失 > 0 的字段）', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
_save(fig, 'fig24_missing_value.png')

# ---------- 高阶图 10：grade 好/坏客户 100% 堆叠占比 ----------
grade_ct = df_raw.groupby('grade')[TARGET].value_counts().unstack().fillna(0)
grade_ct_pct = grade_ct.div(grade_ct.sum(axis=1), axis=0) * 100
grade_ct_pct = grade_ct_pct.sort_index()
fig, ax = plt.subplots(figsize=(7.5, 4.8))
grades = list(grade_ct_pct.index)
good_pct = grade_ct_pct[0].values
bad_pct = grade_ct_pct[1].values
ax.bar(grades, good_pct, color='#1baf7a', label='好客户', width=0.62)
ax.bar(grades, bad_pct, bottom=good_pct, color='#e34948', label='坏客户', width=0.62)
for i, (g, b) in enumerate(zip(good_pct, bad_pct)):
    ax.text(i, g / 2, f'{g:.0f}%', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
    ax.text(i, g + b / 2, f'{b:.0f}%', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
ax.set_xlabel('grade（A 最优 → G 最差）')
ax.set_ylabel('占比 %')
ax.set_title('各等级好/坏客户占比（100% 堆叠，坏客户占比随等级单调上升）', fontsize=13)
ax.legend(frameon=False, ncol=2)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
_save(fig, 'fig25_grade_stacked.png')

# ---------- 高阶图 11：分数分档坏账率（渐变 + 样本数） ----------
decile_sorted = decile.sort_values('分数档')
fig, ax = plt.subplots(figsize=(7.5, 5.0))
vmax_dec = decile_sorted['坏账率'].max()
for i, v in enumerate(decile_sorted['坏账率']):
    _vgradient_bar(ax, i, 0.6, 0, v, cmap_red_ramp, vmin=0, vmax=vmax_dec, n=80)
    ax.text(i, v + 0.6, f'{v:.1f}%', ha='center', fontsize=9, color=INK)
ax2 = ax.twinx()
ax2.plot(np.arange(len(decile_sorted)), decile_sorted['样本数'], color='#184f95', lw=1.8, marker='o', ms=4, label='样本数')
ax2.set_ylabel('样本数', color='#184f95')
ax2.tick_params(axis='y', colors='#184f95')
ax.set_xticks(range(len(decile_sorted)))
ax.set_xticklabels([f'{int(r)}档' for r in decile_sorted['分数档']], rotation=0)
ax.set_xlabel('分数档（1=最低分/最高风险 → 10=最高分/最低风险）')
ax.set_ylabel('坏账率 %')
ax.set_ylim(0, vmax_dec * 1.18)
ax.set_title('分数分档坏账率（单调递减 = 有效区分）', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
_save(fig, 'fig26_decile_gradient.png')

# ---------- 高阶图 12：IV × 相关性气泡图 ----------
corr_sel = X_train.corr().abs()
max_corr = corr_sel.where(~np.eye(len(corr_sel), dtype=bool)).max(axis=1)
bubble = pd.DataFrame({'特征': selected,
                       'IV': [iv_map[c] for c in selected],
                       'max|corr|': [max_corr[c] for c in selected],
                       '|coef|': [abs(coef[c]) for c in selected]})
fig, ax = plt.subplots(figsize=(9, 6.5))
norm_bub = Normalize(bubble['IV'].min(), bubble['IV'].max())
sc = ax.scatter(bubble['IV'], bubble['max|corr|'], s=bubble['|coef|'] * 260 + 40,
                c=bubble['IV'], cmap=cmap_aurora, alpha=0.85, edgecolors='white',
                linewidths=0.6, norm=norm_bub)
for _, r in bubble.iterrows():
    ax.annotate(_cn(r['特征']), (r['IV'], r['max|corr|']),
                textcoords='offset points', xytext=(5, 5), fontsize=8, color=INK)
ax.set_xlabel('IV（信息价值，越大越强）')
ax.set_ylabel('与其它特征最大 |WOE 相关系数|')
ax.set_title('特征区分力 vs 共线性（气泡大小=模型系数强度）', fontsize=13)
cbar = fig.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label('IV')
ax.axhline(0.9, color='#e34948', ls='--', lw=1, label='共线性阈值 0.9')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)
plt.tight_layout()
_save(fig, 'fig27_iv_corr_bubble.png')

print('高阶渐变图表已生成：')
for f in [
    'fig16_roc_gradient.png',
    'fig17_ks_gradient.png',
    'fig18_score_goodbad_kde.png',
    'fig19_scorecard_heatmap.png',
    'fig20_woe_lattice.png',
    'fig21_radar_metrics.png',
    'fig22_lift_curve.png',
    'fig23_model_compare.png',
    'fig24_missing_value.png',
    'fig25_grade_stacked.png',
    'fig26_decile_gradient.png',
    'fig27_iv_corr_bubble.png'
]:
    print('  ', f)
print('=' * 70)


① 业务理解与数据探索（EDA）
数据形状: (50000, 31)
目标变量 y：1=违约(坏) 0=履约(好)
y
好客户    40373
坏客户     9627
坏账率: 19.25%

缺失值一览（全部 31 列的缺失率，含缺失为 0 的列）：
                               缺失数   缺失率%
loan_amnt                        0   0.00
funded_amnt                      0   0.00
term                             0   0.00
int_rate                         0   0.00
installment                      0   0.00
grade                            0   0.00
sub_grade                        0   0.00
emp_length                       0   0.00
home_ownership                   0   0.00
annual_inc                       0   0.00
verification_status              0   0.00
purpose                          0   0.00
addr_state                       0   0.00
dti                              0   0.00
delinq_2yrs                      0   0.00
inq_last_6mths                   0   0.00
mths_since_last_delinq       28097  56.19
open_acc                         0   0.00
pub_rec                          0   0.00
revol_bal                      